# Week 7: Gold set, ungrounded baseline, scoring rubric and held-out checks (Colab version)

Self-contained: choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
It writes the project code, collects the 38 official pages live, builds the gold set (checking every evidence phrase against the live pages), searches all 300 held-out questions and generates grounded and ungrounded answers for an 8-topic subset (`--quick`). Remove `--quick` to answer all 300 (several hours).

Downloads four models (about 15 GB). Pages are fetched live, so numbers can differ slightly from the 16 Nov 2026 run, and the build stops if a page no longer contains an evidence phrase. Sections 3-7 describe the full local run; the quick run does not write the scoring sheet.

In [ ]:
%pip install -q -U transformers sentence-transformers faiss-cpu rank-bm25 lingua-language-detector sentencepiece textstat openpyxl

In [ ]:
import os, sys
for d in ['src', 'eval', 'data/raw', 'data/processed', 'results']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/sources.py
"""Curated list of official sources for the three domains.

Every entry was checked against the live site on 28 Oct 2026 (HTTP 200).
GOV.UK pages are fetched through the public Content API; NHS pages as HTML.
Both are published under the Open Government Licence v3.0, and NHS reuse
also needs attribution, a link back and a date notice (nhs.uk terms).
"""

GOVUK = {
    "schools": [
        "schools-admissions",
        "apply-for-primary-school-place",
        "apply-for-secondary-school-place",
        "school-attendance-absence",
        "apply-free-school-meals",
        "children-with-special-educational-needs",
        "free-school-transport",
        "school-uniform",
        "types-of-school",
        "school-term-holiday-dates",
        "know-when-you-can-leave-school",
        "complain-about-school",
    ],
    "housing": [
        "private-renting",
        "private-renting-evictions",
        "guidance/renters-rights-act-overview-for-tenants",
        "tenancy-deposit-protection",
        "council-housing",
        "housing-association-homes",
        "council-housing-association-evictions",
        "housing-benefit",
        "universal-credit",
        "homelessness-help-from-council",
        "council-tax",
        "apply-council-tax-reduction",
    ],
    "health": [
        "help-nhs-costs",
        "healthcare-immigration-application",
    ],
}

NHS = {
    "health": [
        "https://www.nhs.uk/nhs-services/gps/how-to-register-with-a-gp-surgery/",
        "https://www.nhs.uk/nhs-services/gps/gp-appointments-and-bookings/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-use-111/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-go-to-ae/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-call-999/",
        "https://www.nhs.uk/nhs-services/dentists/how-to-find-an-nhs-dentist/",
        "https://www.nhs.uk/nhs-services/dentists/how-much-nhs-dental-treatment-costs/",
        "https://www.nhs.uk/nhs-services/pharmacies/how-pharmacies-can-help/",
        "https://www.nhs.uk/nhs-services/help-with-health-costs/nhs-low-income-scheme-lis/",
        "https://www.nhs.uk/nhs-services/visiting-or-moving-to-england/how-to-access-nhs-services-in-england-if-you-are-visiting-from-abroad/",
        "https://www.nhs.uk/vaccinations/nhs-vaccinations-and-when-to-have-them/",
        "https://www.nhs.uk/pregnancy/finding-out/what-to-do-when-you-find-out-you-are-pregnant/",
    ],
}


In [ ]:
%%writefile src/ingest.py
"""Fetch every source page, keep a raw snapshot, and write one clean record per page.

Output: data/raw/<id>.(json|html) and data/processed/pages.jsonl
Each record keeps the section structure (h2/h3 headings) so chunks can carry
their heading, plus three dates: the publisher's own date, the publisher's
last technical update, and when we fetched it (for data-currency reporting).
"""
import hashlib
import json
import pathlib
import time
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup

from sources import GOVUK, NHS

ROOT = pathlib.Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "pages.jsonl"
UA = {"User-Agent": "MSc research prototype (multilingual civic assistant); polite crawler"}

# GOV.UK formats store text in different fields; order follows the rendered page.
GOVUK_TEXT_FIELDS = ["introduction", "introductory_paragraph", "body", "more_information",
                     "need_to_know", "other_ways_to_apply", "what_you_need_to_know"]


def page_id(url):
    return hashlib.sha1(url.encode()).hexdigest()[:10]


def sections_from_html(html, page_title):
    """Split HTML into (heading, text) sections at h2/h3 boundaries."""
    soup = BeautifulSoup(html, "html.parser")
    sections, heading, buf = [], page_title, []
    for el in soup.find_all(["h2", "h3", "p", "li", "td", "th"]):
        if el.name in ("h2", "h3"):
            if buf:
                sections.append({"heading": heading, "text": " ".join(buf)})
            heading, buf = el.get_text(" ", strip=True), []
        elif not el.find_parent(["li", "td", "th"]) or el.name in ("td", "th"):
            # skip nested elements already covered by their parent's text
            t = el.get_text(" ", strip=True)
            if t:
                buf.append(t)
    if buf:
        sections.append({"heading": heading, "text": " ".join(buf)})
    return sections


def fetch_govuk(path, domain):
    r = requests.get("https://www.gov.uk/api/content/" + path, headers=UA, timeout=30)
    r.raise_for_status()
    data = r.json()
    url = "https://www.gov.uk/" + path
    (RAW / f"{page_id(url)}.json").write_text(r.text)
    det = data.get("details", {})
    html = "".join(det.get(f, "") for f in GOVUK_TEXT_FIELDS if isinstance(det.get(f), str))
    for part in det.get("parts", []):
        html += f"<h2>{part['title']}</h2>{part['body']}"
    title = data.get("title", "")
    return {
        "id": page_id(url), "url": url, "source": "GOV.UK", "domain": domain,
        "title": title, "format": data.get("schema_name"),
        "public_updated_at": (data.get("public_updated_at") or "")[:10],
        "updated_at": (data.get("updated_at") or "")[:10],
        "sections": sections_from_html(html, title),
    }


def fetch_nhs(url, domain):
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    (RAW / f"{page_id(url)}.html").write_text(r.text)
    soup = BeautifulSoup(r.text, "html.parser")
    reviewed = soup.find(string=lambda s: s and "Page last reviewed" in s)
    reviewed = reviewed.strip().split(":", 1)[1].strip() if reviewed else ""
    modified = ""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            modified = (json.loads(s.string or "{}").get("dateModified") or modified)[:10]
        except (ValueError, AttributeError):
            pass
    article = soup.find("article") or soup.find("main")
    for junk in article.find_all(["aside", "nav", "footer", "script", "style"]):
        junk.decompose()
    title = soup.find("h1").get_text(" ", strip=True)
    # the lead sentence ("999 is for life-threatening emergencies...") sits outside <article>
    lede = soup.find(class_="nhsuk-lede-text")
    html = (str(lede) if lede and article not in lede.parents else "") + str(article)
    return {
        "id": page_id(url), "url": url, "source": "NHS.uk", "domain": domain,
        "title": title, "format": "nhs_article",
        "public_updated_at": datetime.strptime(reviewed, "%d %B %Y").strftime("%Y-%m-%d") if reviewed else "",
        "updated_at": modified,
        "sections": sections_from_html(html, title),
    }


def main():
    RAW.mkdir(parents=True, exist_ok=True)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    jobs = [(fetch_govuk, p, d) for d, ps in GOVUK.items() for p in ps]
    jobs += [(fetch_nhs, u, d) for d, us in NHS.items() for u in us]
    records, failed = [], []
    for fn, target, domain in jobs:
        try:
            rec = fn(target, domain)
            rec["fetched_at"] = fetched_at
            rec["words"] = sum(len(s["text"].split()) for s in rec["sections"])
            records.append(rec)
            print(f"ok   {rec['domain']:8} {rec['words']:5}w  {rec['title']}")
        except Exception as e:  # keep going; report at the end
            failed.append((target, repr(e)))
            print(f"FAIL {target}: {e}")
        time.sleep(0.5)
    with OUT.open("w") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"\n{len(records)} pages written to {OUT.relative_to(ROOT)}; {len(failed)} failed")
    for t, e in failed:
        print("  failed:", t, e)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/chunk.py
"""Turn pages into retrievable chunks that respect the page's own sections.

Rules (kept simple so they are easy to describe in the methodology):
  * one chunk per h2/h3 section;
  * each section is tagged with the UK nation it applies to (from its heading);
  * sections shorter than MIN_WORDS are merged into the next one on the same page,
    but only when both have the same nation tag;
  * sections longer than MAX_WORDS are split into windows with OVERLAP words;
  * every chunk text starts with "<page title> - <section heading>:" so the
    embedding sees the context a reader would see.
Output: data/processed/chunks.csv
"""
import json
import pathlib
import re

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
MIN_WORDS, MAX_WORDS, OVERLAP = 40, 200, 40
OTHER_NATIONS = ["Scotland", "Wales", "Northern Ireland"]


def nation_of(heading):
    """GOV.UK marks nation-specific sections in the heading ("Scotland", "If you're in Wales...").
    Everything else applies to England (or the whole UK). NHS.uk pages cover England only."""
    named = [n for n in OTHER_NATIONS if re.search(rf"\b{n}\b", heading)]
    return ";".join(n.lower().replace(" ", "_") for n in named) or "england"


def join(a, b):
    return {"heading": a["heading"] + " / " + b["heading"], "text": a["text"] + " " + b["text"],
            "nation": a["nation"]}


def merge_small(sections):
    merged, carry = [], None
    for s in sections:
        s = {**s, "nation": nation_of(s["heading"])}
        if carry and carry["nation"] == s["nation"]:
            s = join(carry, s)
        elif carry:
            merged.append(carry)  # never merge across nations; keep the short one alone
        carry = None
        if len(s["text"].split()) < MIN_WORDS:
            carry = s
        else:
            merged.append(s)
    if carry:  # trailing short section joins the previous one if same nation, or stands alone
        if merged and merged[-1]["nation"] == carry["nation"]:
            merged[-1] = join(merged[-1], carry)
        else:
            merged.append(carry)
    return merged


def windows(words):
    if len(words) <= MAX_WORDS:
        return [words]
    step = MAX_WORDS - OVERLAP
    return [words[i:i + MAX_WORDS] for i in range(0, len(words) - OVERLAP, step)]


def main():
    pages = [json.loads(l) for l in open(ROOT / "data/processed/pages.jsonl")]
    rows = []
    for p in pages:
        for s_i, s in enumerate(merge_small(p["sections"])):
            for w_i, w in enumerate(windows(s["text"].split())):
                body = " ".join(w)
                rows.append({
                    "chunk_id": f"{p['id']}-{s_i:02d}-{w_i}",
                    "page_id": p["id"], "url": p["url"], "source": p["source"], "domain": p["domain"],
                    "title": p["title"], "heading": s["heading"], "nation": s["nation"],
                    "public_updated_at": p["public_updated_at"], "fetched_at": p["fetched_at"],
                    "words": len(w), "body": body,
                    "text": f"{p['title']} - {s['heading']}: {body}",
                })
    df = pd.DataFrame(rows)
    df.to_csv(ROOT / "data/processed/chunks.csv", index=False)
    print(f"{len(pages)} pages -> {len(df)} chunks")
    print(df.groupby("domain").agg(pages=("page_id", "nunique"), chunks=("chunk_id", "count"),
                                   words=("words", "sum")).to_string())
    print("nation tags:", df.nation.value_counts().to_dict())
    print("chunk words: median", int(df.words.median()), "min", df.words.min(), "max", df.words.max())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/retrieval.py
"""Dense and lexical retrievers over the chunk table.

Dense: sentence-transformers model + FAISS inner-product index on normalised
vectors (= cosine similarity). E5 models need "query: " / "passage: " prefixes.
Lexical: BM25 (rank_bm25), used as a non-neural baseline.
"""
import re

import torch  # noqa: F401  must load before faiss: on macOS the two OpenMP runtimes otherwise crash NLLB generation
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


class DenseRetriever:
    def __init__(self, model_name, texts):
        self.model = SentenceTransformer(model_name)
        self.e5 = "e5" in model_name.lower()
        passages = [("passage: " + t) if self.e5 else t for t in texts]
        emb = self.model.encode(passages, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(np.asarray(emb, dtype="float32"))

    def search(self, query, k=10):
        q = self.model.encode([("query: " + query) if self.e5 else query], normalize_embeddings=True)
        scores, ids = self.index.search(np.asarray(q, dtype="float32"), k)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    def save(self, path):
        faiss.write_index(self.index, str(path))


def tokenize(text):
    return re.findall(r"\w+", text.lower())


class BM25Retriever:
    def __init__(self, texts):
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])

    def search(self, query, k=10):
        scores = self.bm25.get_scores(tokenize(query))
        ids = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in ids]


In [ ]:
%%writefile src/language.py
"""Detect which supported language a question is written in.

Uses lingua (Apache-2.0), restricted to the five project languages, which makes
short questions much easier to classify than open-set detection. Arabic and Urdu
share a script, so restricting the set matters most for those two.
"""
from lingua import Language, LanguageDetectorBuilder

SUPPORTED = {
    Language.ENGLISH: "en", Language.POLISH: "pl", Language.ROMANIAN: "ro",
    Language.ARABIC: "ar", Language.URDU: "ur",
}
_detector = LanguageDetectorBuilder.from_languages(*SUPPORTED).build()


def detect(text):
    """Return 'en', 'pl', 'ro', 'ar' or 'ur'; falls back to 'en' if lingua cannot decide."""
    lang = _detector.detect_language_of(text)
    return SUPPORTED.get(lang, "en")


In [ ]:
%%writefile src/translate.py
"""Machine translation with NLLB-200 (facebook/nllb-200-distilled-600M, CC-BY-NC-4.0).

Used in week 5 to translate questions into English before retrieval, and in week 6
for answers going back into the user's language. Non-commercial licence: fine for
this research prototype, not for a production service.
"""
import re

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL = "facebook/nllb-200-distilled-600M"
NLLB_CODES = {"en": "eng_Latn", "pl": "pol_Latn", "ro": "ron_Latn", "ar": "arb_Arab", "ur": "urd_Arab"}


class Translator:
    def __init__(self, model_name=MODEL):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU on Colab, CPU elsewhere
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).eval().to(self.device)

    @torch.inference_mode()
    def _translate_sentences(self, sentences, src, tgt, max_length):
        self.tok.src_lang = NLLB_CODES[src]
        batch = self.tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=400).to(self.device)
        out = self.model.generate(
            **batch, max_length=max_length, num_beams=4,
            forced_bos_token_id=self.tok.convert_tokens_to_ids(NLLB_CODES[tgt]))
        return self.tok.batch_decode(out, skip_special_tokens=True)

    def translate(self, texts, src, tgt="en", max_length=256):
        """Translate a list of strings from language `src` to `tgt` (project codes, e.g. 'pl').

        NLLB is trained on single sentences and silently drops later sentences of a
        multi-sentence input, so each text is split into sentences first and rejoined.
        """
        if src == tgt:
            return list(texts)
        split = [split_sentences(t) for t in texts]
        flat = [s for parts in split for s in parts]
        done = iter(self._translate_sentences(flat, src, tgt, max_length))
        return [" ".join(next(done) for _ in parts) for parts in split]


def split_sentences(text):
    """Split on sentence-final punctuation, including Arabic ؟ and Urdu ۔ ."""
    parts = [p.strip() for p in re.split(r"(?<=[.!?؟۔])\s+", text.strip())]
    return [p for p in parts if p] or [text]


In [ ]:
%%writefile src/rerank.py
"""Second-stage reranking with a multilingual cross-encoder (BAAI/bge-reranker-v2-m3, Apache-2.0).

The bi-encoder (e5) compares a question vector with pre-computed chunk vectors, which
is fast but coarse. A cross-encoder reads the question and one chunk together and
scores how well that chunk answers it; it is too slow for the whole corpus, so it
only reorders the top candidates from the first stage.
"""
from sentence_transformers import CrossEncoder

MODEL = "BAAI/bge-reranker-v2-m3"


class Reranker:
    def __init__(self, model_name=MODEL):
        self.model = CrossEncoder(model_name, max_length=512)

    def rerank(self, query, candidates, texts):
        """candidates: list of chunk ids; texts: chunk id -> text. Returns [(id, score)] best first."""
        if not candidates:
            return []
        scores = self.model.predict([(query, texts[i]) for i in candidates], batch_size=16, show_progress_bar=False)
        return sorted(zip(candidates, map(float, scores)), key=lambda x: -x[1])


In [ ]:
%%writefile src/pipeline.py
"""Week 5 search pipeline: question in any supported language -> best official passages.

    question
      -> detect language                       (language.py)
      -> translate to English if not English   (translate.py, optional)
      -> dense retrieval, original + English   (retrieval.py, e5 + FAISS)
      -> fuse the two ranked lists (RRF)
      -> keep England / UK-wide chunks only     (nation tag from chunk.py)
      -> rerank top candidates                  (rerank.py, optional)
      -> top k passages with title, heading, URL and dates for citation
"""
import pathlib

import pandas as pd

from language import detect
from retrieval import DenseRetriever

ROOT = pathlib.Path(__file__).resolve().parents[1]
E5 = "intfloat/multilingual-e5-base"
RRF_K = 60  # standard constant from Cormack et al. (2009)


def rrf(*ranked_lists, k=RRF_K):
    """Reciprocal rank fusion: combine ranked lists of ids into one ranking."""
    score = {}
    for ranked in ranked_lists:
        for rank, i in enumerate(ranked, start=1):
            score[i] = score.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=lambda i: -score[i])


class SearchPipeline:
    def __init__(self, chunks=None, translate=True, rerank=True, england_only=True, pool=20):
        self.chunks = chunks if chunks is not None else pd.read_csv(ROOT / "data/processed/chunks.csv")
        self.texts = self.chunks.text.tolist()
        self.retriever = DenseRetriever(E5, self.texts)
        self.allowed = set(self.chunks.index[self.chunks.nation == "england"]) if england_only else None
        self.pool = pool
        self.translator = None
        if translate:
            from translate import Translator
            self.translator = Translator()
        self.reranker = None
        if rerank:
            from rerank import Reranker
            self.reranker = Reranker()

    def _dense(self, query, n):
        return [i for i, _ in self.retriever.search(query, k=n)]

    def search(self, question, k=3, lang=None):
        lang = lang or detect(question)
        english = question
        lists = [self._dense(question, 50)]
        if self.translator and lang != "en":
            english = self.translator.translate([question], src=lang)[0]
            lists.append(self._dense(english, 50))
        ranked = rrf(*lists) if len(lists) > 1 else lists[0]
        if self.allowed is not None:
            ranked = [i for i in ranked if i in self.allowed]
        ranked = ranked[:self.pool]
        scores = {}
        if self.reranker:
            scores = dict(self.reranker.rerank(question, ranked, self.texts))
            ranked = sorted(ranked, key=lambda i: -scores[i])
        results = []
        for i in ranked[:k]:
            c = self.chunks.iloc[i]
            results.append({"chunk": int(i), "title": c.title, "heading": c.heading, "url": c.url,
                            "source": c.source, "nation": c.nation, "public_updated_at": c.public_updated_at,
                            "fetched_at": c.fetched_at, "text": c.body,
                            "score": scores.get(i)})  # reranker relevance (0-1), used by the refusal rule
        return {"lang": lang, "english_query": english, "results": results}


In [ ]:
%%writefile src/generate.py
"""Grounded answer generation with an openly licensed LLM (Qwen/Qwen3.5-4B, Apache-2.0).

Why this model (task T26):
  * Apache-2.0 and not gated, so it can be run locally or on a free Colab GPU and user questions
    never leave the machine (see docs/ethics-notes.md: no third-party API without consent).
  * The model card states support for 201 languages and dialects; Polish, Romanian, Arabic and Urdu
    are all needed here.
  * 4B parameters (9.3 GB of weights in 16-bit): fits a 16 GB Colab T4 in float16 and the 24 GB development laptop in bfloat16.
Qwen3.5 "thinks" before answering by default; thinking is switched off (enable_thinking=False) so the
output is the answer only. Decoding is greedy so that every run gives the same answer.
"""
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3.5-4B"
NO_ANSWER = "NO_ANSWER"
LANGUAGE_NAMES = {"en": "English", "pl": "Polish", "ro": "Romanian", "ar": "Arabic", "ur": "Urdu"}

SYSTEM = """You help people in England understand official guidance on health services, housing and schools.
You are given numbered extracts from official GOV.UK and NHS.uk pages, and a question.

Rules:
1. Use ONLY facts stated in the extracts. Never add facts, dates, places or general remarks from your own knowledge.
2. Every sentence must end with the number of the extract it comes from, in square brackets, for example [1] or [1][2].
   If you cannot point to an extract for a sentence, leave that sentence out.
3. Answer the question first. Write in plain language: short sentences, everyday words, no more than 120 words.
4. If the extracts do not answer the question, reply with exactly NO_ANSWER and nothing else.
5. Do not give medical diagnoses or legal advice about the user's own case; explain what the guidance says.
6. Write the whole answer in {language}."""

# Ungrounded baseline (task T30): same model, same audience and style rules, but no extracts and no citations.
BASELINE = """You help people in England understand official guidance on health services, housing and schools.

Rules:
1. Answer the question first. Write in plain language: short sentences, everyday words, no more than 120 words.
2. If you do not know the answer, say so.
3. Do not give medical diagnoses or legal advice about the user's own case; explain what the guidance says.
4. Write the whole answer in {language}."""

CHECK = """You check whether numbered extracts from official GOV.UK and NHS.uk pages answer a question.
The question may be in any language. Reply YES if the extracts contain the information needed to answer it,
even partly. Reply NO if they do not. Reply with one word: YES or NO."""

CITATION = re.compile(r"\[(\d+)\]")


def format_sources(passages):
    """Number the passages [1]..[k] the way the model will cite them."""
    return "\n\n".join(f"[{n}] {p['title']} - {p['heading']}\n{p['text']}" for n, p in enumerate(passages, 1))


def build_messages(question, passages, answer_lang, pivot=False):
    """pivot=True: the answer is written in English for a question in another language (translate mode).
    The model tends to follow the question's language, so the instruction is repeated after the question."""
    system = SYSTEM.format(language=LANGUAGE_NAMES[answer_lang])
    user = f"Extracts:\n\n{format_sources(passages)}\n\nQuestion: {question}"
    if pivot:
        user += "\n\nWrite the answer in English, even though the question is in another language."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def check_messages(question, passages):
    user = f"Extracts:\n\n{format_sources(passages)}\n\nQuestion: {question}\n\nDo the extracts answer the question?"
    return [{"role": "system", "content": CHECK}, {"role": "user", "content": user}]


def baseline_messages(question, answer_lang):
    return [{"role": "system", "content": BASELINE.format(language=LANGUAGE_NAMES[answer_lang])},
            {"role": "user", "content": question}]


def citations(answer):
    """Citation numbers used in an answer, in order of first use."""
    return list(dict.fromkeys(int(n) for n in CITATION.findall(answer)))


class Generator:
    def __init__(self, model_name=MODEL, device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.bfloat16 if device == "mps" else torch.float32
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(device).eval()

    @torch.inference_mode()
    def chat(self, messages, max_new_tokens=300):
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                    enable_thinking=False)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return self.tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def answerable(self, question, passages):
        """Refusal rule 2: a separate yes/no call - do the passages answer the question?"""
        return self.chat(check_messages(question, passages), max_new_tokens=3).strip().upper().startswith("YES")

    def answer_ungrounded(self, question, answer_lang):
        """Baseline: the same model answers from its own knowledge, with no retrieved sources."""
        return self.chat(baseline_messages(question, answer_lang))

    def answer(self, question, passages, answer_lang, pivot=False):
        """Answer from the passages only. Returns the raw model text (may be NO_ANSWER)."""
        return self.chat(build_messages(question, passages, answer_lang, pivot))


In [ ]:
%%writefile src/assistant.py
"""Week 6 assistant: question in any supported language -> cited answer from official pages, or a refusal.

    question
      -> search stage (pipeline.py, set-up F from week 5): top 3 passages + reranker score
      -> refusal rule 1: top reranker score below THRESHOLD -> refuse without calling the LLM
      -> refusal rule 2: a separate yes/no LLM call says the passages do not answer it -> refuse
      -> LLM answers only from the 3 passages, citing [n]          (generate.py)
      -> refusal rule 3: the LLM replies NO_ANSWER -> refuse
      -> answer language: "direct"    = the LLM writes in the user's language
                          "translate" = the LLM writes in English, NLLB translates it
      -> source cards for the cited passages (title, section, link, date fetched)
      -> health note (NHS 111 / 999) whenever an NHS.uk page is cited

Citations are numbers chosen by the LLM; titles, links and dates always come from the corpus, so the
model can never invent a URL.
"""
from generate import NO_ANSWER, Generator, citations, CITATION
from pipeline import SearchPipeline
from translate import split_sentences

# Calibrated in eval/evaluate_week6.py on 150 answerable and 50 unanswerable questions.
THRESHOLD = 0.0052  # 0.9 x lowest top score of an answerable question whose answer was retrieved

# Fixed messages (written by the author; part of the native-speaker check, task T22).
REFUSAL = {
    "en": "I could not find an answer to this in the official GOV.UK and NHS pages I use. "
          "Please check GOV.UK or NHS.uk, or contact your local council.",
    "pl": "Nie udało mi się znaleźć odpowiedzi na to pytanie na oficjalnych stronach GOV.UK i NHS. "
          "Sprawdź GOV.UK lub NHS.uk albo skontaktuj się z lokalnym urzędem (council).",
    "ro": "Nu am găsit un răspuns la această întrebare în paginile oficiale GOV.UK și NHS. "
          "Vă rugăm să verificați GOV.UK sau NHS.uk ori să contactați consiliul local.",
    "ar": "لم أجد إجابة عن هذا السؤال في صفحات GOV.UK وNHS الرسمية. "
          "يُرجى مراجعة GOV.UK أو NHS.uk، أو التواصل مع المجلس المحلي.",
    "ur": "مجھے سرکاری GOV.UK اور NHS صفحات میں اس سوال کا جواب نہیں ملا۔ "
          "براہ کرم GOV.UK یا NHS.uk دیکھیں، یا اپنی مقامی کونسل سے رابطہ کریں۔",
}
HEALTH_NOTE = {
    "en": "If you need medical help now, call 111. In an emergency, call 999.",
    "pl": "Jeśli potrzebujesz pilnej pomocy medycznej, zadzwoń pod numer 111. W nagłym wypadku dzwoń pod 999.",
    "ro": "Dacă aveți nevoie urgent de ajutor medical, sunați la 111. În caz de urgență, sunați la 999.",
    "ar": "إذا كنت بحاجة إلى مساعدة طبية الآن، اتصل بالرقم 111. في حالات الطوارئ، اتصل بالرقم 999.",
    "ur": "اگر آپ کو ابھی طبی مدد کی ضرورت ہے تو 111 پر کال کریں۔ ہنگامی صورت میں 999 پر کال کریں۔",
}


def translate_answer(translator, english, lang):
    """Translate an English answer sentence by sentence, keeping each sentence's [n] citations.

    Citation markers are taken out before translation (NLLB may drop or garble them) and put back
    at the end of the translated sentence."""
    out = []
    for sent in split_sentences(english):
        marks = "".join(f"[{n}]" for n in CITATION.findall(sent))
        text = CITATION.sub("", sent).replace(" .", ".").strip()
        if text:
            out.append((translator.translate([text], src="en", tgt=lang)[0] + " " + marks).strip())
        elif marks and out:  # markers split off on their own belong to the previous sentence
            out[-1] += " " + marks
    return " ".join(out)


class Assistant:
    def __init__(self, pipeline=None, generator=None, threshold=THRESHOLD, mode="direct"):
        self.pipeline = pipeline or SearchPipeline()
        self.generator = generator or Generator()
        self.threshold = threshold
        self.mode = mode

    def refuse(self, base, reason):
        return {**base, "refused": True, "reason": reason, "answer": REFUSAL[base["lang"]], "sources": []}

    def answer_from(self, search, question, mode=None):
        """Answer from an existing search result (lets the evaluation reuse one search for both modes)."""
        mode = mode or self.mode
        lang, passages = search["lang"], search["results"]
        top = passages[0]["score"] if passages else 0.0
        base = {"question": question, "lang": lang, "mode": mode, "english_query": search["english_query"],
                "top_score": top, "raw": ""}
        if top < self.threshold:
            return self.refuse(base, "low relevance")
        if not self.generator.answerable(question, passages):
            return self.refuse(base, "check: not in sources")
        answer_lang = lang if mode == "direct" else "en"
        raw = self.generator.answer(question, passages, answer_lang, pivot=answer_lang != lang)
        base["raw"] = raw
        if NO_ANSWER in raw:
            return self.refuse(base, "model: not in sources")
        text = raw if answer_lang == lang else translate_answer(self.pipeline.translator, raw, lang)
        cited = [n for n in citations(text) if 1 <= n <= len(passages)]
        sources = [{"n": n, **{k: passages[n - 1][k] for k in ("title", "heading", "url", "source", "fetched_at")}}
                   for n in cited]
        if any(s["source"] == "NHS.uk" for s in sources):
            text += "\n\n" + HEALTH_NOTE[lang]
        return {**base, "refused": False, "reason": None, "answer": text, "sources": sources,
                "passages": passages}

    def ask(self, question, lang=None, mode=None):
        return self.answer_from(self.pipeline.search(question, k=3, lang=lang), question, mode)


In [ ]:
%%writefile eval/build_queries.py
"""Build the retrieval test set and check every answer phrase against the corpus.

Each English question has:
  gold_url - the page that answers it
  evidence - a short phrase copied verbatim from that page that contains the answer
A retrieved chunk only counts as "answer-bearing" if it contains the evidence
phrase, so a hit on the right page but the wrong section is not counted.

Translations (pl, ro, ar, ur) were written by the author and are flagged
"unverified" until a native speaker checks them.
Output: eval/queries.csv
"""
import csv
import json
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
G = "https://www.gov.uk/"
N = "https://www.nhs.uk/nhs-services/"

# qid: (domain, gold_url, evidence, {lang: question})
Q = {
    "S01": ("schools", G + "schools-admissions", "close on 15 January", {
        "en": "When is the deadline to apply for a primary school place?",
        "pl": "Jaki jest termin składania wniosku o miejsce w szkole podstawowej?",
        "ro": "Care este termenul limită pentru a depune cererea pentru un loc la școala primară?",
        "ar": "ما هو الموعد النهائي للتقديم على مقعد في المدرسة الابتدائية؟",
        "ur": "پرائمری اسکول میں داخلے کی درخواست دینے کی آخری تاریخ کیا ہے؟"}),
    "S02": ("schools", G + "schools-admissions", "by 31 October", {
        "en": "What is the last date to apply for a secondary school place?",
        "pl": "Do kiedy trzeba złożyć wniosek o miejsce w szkole średniej?",
        "ro": "Până la ce dată trebuie să aplic pentru un loc la școala secundară?",
        "ar": "ما هو آخر موعد للتقديم على مقعد في المدرسة الثانوية؟",
        "ur": "سیکنڈری اسکول میں داخلے کے لیے درخواست کس تاریخ تک دینی ہوگی؟"}),
    "S03": ("schools", G + "schools-admissions", "primary schools on 16 April", {
        "en": "When will I find out which primary school my child has been offered?",
        "pl": "Kiedy dowiem się, do której szkoły podstawowej zostało przyjęte moje dziecko?",
        "ro": "Când voi afla la ce școală primară a primit loc copilul meu?",
        "ar": "متى سأعرف المدرسة الابتدائية التي قُبل فيها طفلي؟",
        "ur": "مجھے کب پتہ چلے گا کہ میرے بچے کو کون سے پرائمری اسکول میں جگہ ملی ہے؟"}),
    "S04": ("schools", G + "schools-admissions", "at least 20 school days", {
        "en": "How long do I have to appeal if my child is refused a school place?",
        "pl": "Ile mam czasu na odwołanie, jeśli moje dziecko nie dostało miejsca w szkole?",
        "ro": "Cât timp am la dispoziție să fac contestație dacă copilul meu nu a primit loc la școală?",
        "ar": "كم لدي من الوقت للاستئناف إذا رُفض طلب طفلي للحصول على مقعد في المدرسة؟",
        "ur": "اگر میرے بچے کو اسکول میں جگہ نہ ملے تو اپیل کرنے کے لیے میرے پاس کتنا وقت ہے؟"}),
    "S05": ("schools", G + "school-attendance-absence", "fined or face legal action for taking your child on holiday", {
        "en": "Can I be fined for taking my child on holiday during term time?",
        "pl": "Czy mogę dostać grzywnę za zabranie dziecka na wakacje w trakcie roku szkolnego?",
        "ro": "Pot primi amendă dacă îmi iau copilul în vacanță în timpul semestrului?",
        "ar": "هل يمكن أن أُغرَّم إذا أخذت طفلي في عطلة أثناء الفصل الدراسي؟",
        "ur": "کیا ٹرم کے دوران بچے کو چھٹیوں پر لے جانے پر مجھے جرمانہ ہو سکتا ہے؟"}),
    "S06": ("schools", G + "apply-free-school-meals", "free school meals if you get any of the following", {
        "en": "Can my child get free school meals if I receive Universal Credit?",
        "pl": "Czy moje dziecko może dostać darmowe posiłki w szkole, jeśli otrzymuję Universal Credit?",
        "ro": "Poate copilul meu să primească mese gratuite la școală dacă primesc Universal Credit?",
        "ar": "هل يمكن لطفلي الحصول على وجبات مدرسية مجانية إذا كنت أتلقى Universal Credit؟",
        "ur": "اگر مجھے Universal Credit ملتا ہے تو کیا میرے بچے کو اسکول میں مفت کھانا مل سکتا ہے؟"}),
    "S07": ("schools", G + "free-school-transport", "more than 2 miles away and the child is under 8", {
        "en": "Is school transport free if the school is far from home?",
        "pl": "Czy transport do szkoły jest darmowy, jeśli szkoła jest daleko od domu?",
        "ro": "Este transportul la școală gratuit dacă școala este departe de casă?",
        "ar": "هل المواصلات المدرسية مجانية إذا كانت المدرسة بعيدة عن المنزل؟",
        "ur": "اگر اسکول گھر سے دور ہو تو کیا اسکول کی ٹرانسپورٹ مفت ہے؟"}),
    "S08": ("schools", G + "know-when-you-can-leave-school", "16 by the end of the summer holidays", {
        "en": "At what age can my child leave school?",
        "pl": "W jakim wieku moje dziecko może skończyć szkołę?",
        "ro": "La ce vârstă poate copilul meu să termine școala?",
        "ar": "في أي عمر يمكن لطفلي ترك المدرسة؟",
        "ur": "میرا بچہ کس عمر میں اسکول چھوڑ سکتا ہے؟"}),
    "S09": ("schools", G + "children-with-special-educational-needs", "SEN support - support given in school", {
        "en": "What support can my child get for special educational needs?",
        "pl": "Jakie wsparcie może otrzymać moje dziecko ze specjalnymi potrzebami edukacyjnymi?",
        "ro": "Ce sprijin poate primi copilul meu pentru cerințe educaționale speciale?",
        "ar": "ما الدعم الذي يمكن أن يحصل عليه طفلي إذا كانت لديه احتياجات تعليمية خاصة؟",
        "ur": "خصوصی تعلیمی ضروریات والے میرے بچے کو کیا مدد مل سکتی ہے؟"}),
    "S10": ("schools", G + "complain-about-school", "different ways to complain in England", {
        "en": "How do I complain about my child's state school?",
        "pl": "Jak złożyć skargę na państwową szkołę mojego dziecka?",
        "ro": "Cum fac o plângere împotriva școlii de stat a copilului meu?",
        "ar": "كيف أقدم شكوى ضد المدرسة الحكومية التي يدرس فيها طفلي؟",
        "ur": "میں اپنے بچے کے سرکاری اسکول کے بارے میں شکایت کیسے کروں؟"}),

    "H01": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "section 21 process", {
        "en": "Can my landlord evict me without giving a reason?",
        "pl": "Czy wynajmujący może mnie eksmitować bez podania powodu?",
        "ro": "Poate proprietarul să mă evacueze fără să-mi dea un motiv?",
        "ar": "هل يمكن للمالك أن يطردني دون إبداء سبب؟",
        "ur": "کیا مالک مکان مجھے بغیر کوئی وجہ بتائے گھر سے نکال سکتا ہے؟"}),
    "H02": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "increase the rent once a year", {
        "en": "How often can my landlord increase the rent?",
        "pl": "Jak często wynajmujący może podnosić czynsz?",
        "ro": "Cât de des poate proprietarul să crească chiria?",
        "ar": "كم مرة يمكن للمالك أن يرفع الإيجار؟",
        "ur": "مالک مکان کتنی بار کرایہ بڑھا سکتا ہے؟"}),
    "H03": ("housing", G + "tenancy-deposit-protection", "Deposit Protection Service", {
        "en": "My landlord took a deposit from me. Does it have to be protected?",
        "pl": "Wynajmujący pobrał ode mnie kaucję. Czy musi ona być zabezpieczona?",
        "ro": "Proprietarul mi-a luat o garanție. Trebuie aceasta să fie protejată?",
        "ar": "أخذ المالك مني مبلغ تأمين، هل يجب أن يكون هذا التأمين محميًا؟",
        "ur": "مالک مکان نے مجھ سے ڈپازٹ لیا ہے، کیا اس ڈپازٹ کا محفوظ ہونا ضروری ہے؟"}),
    "H04": ("housing", G + "housing-benefit", "help you pay your rent", {
        "en": "How can I get help paying my rent if I am on a low income?",
        "pl": "Jak uzyskać pomoc w opłaceniu czynszu, jeśli mam niskie dochody?",
        "ro": "Cum pot primi ajutor pentru plata chiriei dacă am un venit mic?",
        "ar": "كيف أحصل على مساعدة في دفع الإيجار إذا كان دخلي منخفضًا؟",
        "ur": "کم آمدنی کی صورت میں مجھے کرایہ ادا کرنے میں مدد کیسے مل سکتی ہے؟"}),
    "H05": ("housing", G + "council-housing", "join a waiting list", {
        "en": "How do I apply for council housing?",
        "pl": "Jak złożyć wniosek o mieszkanie komunalne?",
        "ro": "Cum aplic pentru o locuință socială de la consiliul local?",
        "ar": "كيف أتقدم بطلب للحصول على سكن من المجلس المحلي؟",
        "ur": "میں کونسل کے مکان کے لیے درخواست کیسے دوں؟"}),
    "H06": ("housing", G + "homelessness-help-from-council", "homeless or about to lose your home", {
        "en": "I am about to lose my home. Can the council help me?",
        "pl": "Zaraz stracę dom. Czy lokalny urząd może mi pomóc?",
        "ro": "Sunt pe punctul de a-mi pierde locuința. Mă poate ajuta consiliul local?",
        "ar": "أنا على وشك أن أفقد منزلي، هل يمكن للمجلس المحلي مساعدتي؟",
        "ur": "میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟"}),
    "H07": ("housing", G + "private-renting", "safe and in a good state of repair", {
        "en": "Does my landlord have to keep my rented home in good repair?",
        "pl": "Czy wynajmujący musi utrzymywać wynajmowane mieszkanie w dobrym stanie technicznym?",
        "ro": "Este proprietarul obligat să mențină locuința închiriată în stare bună?",
        "ar": "هل يجب على المالك أن يحافظ على المنزل المؤجر في حالة جيدة؟",
        "ur": "کیا مالک مکان پر کرائے کے گھر کو اچھی حالت میں رکھنا لازم ہے؟"}),
    "H08": ("housing", G + "private-renting", "24 hours", {
        "en": "How much notice must my landlord give before visiting the property?",
        "pl": "Z jakim wyprzedzeniem wynajmujący musi mnie uprzedzić przed wizytą w mieszkaniu?",
        "ro": "Cu cât timp înainte trebuie să mă anunțe proprietarul dacă vrea să vină în locuință?",
        "ar": "كم من الوقت يجب أن يخطرني المالك قبل أن يزور المنزل؟",
        "ur": "گھر آنے سے پہلے مالک مکان کو مجھے کتنا پہلے اطلاع دینی چاہیے؟"}),
    "H09": ("housing", G + "council-tax", "25% off your bill", {
        "en": "Do I get a council tax discount if I live alone?",
        "pl": "Czy dostanę zniżkę na podatek lokalny, jeśli mieszkam sam?",
        "ro": "Primesc reducere la impozitul local pe locuință dacă locuiesc singur?",
        "ar": "هل أحصل على خصم في ضريبة المجلس المحلي إذا كنت أعيش وحدي؟",
        "ur": "اگر میں اکیلا رہتا ہوں تو کیا مجھے کونسل ٹیکس میں رعایت ملتی ہے؟"}),
    "H10": ("housing", G + "council-housing-association-evictions", "stop or delay the eviction", {
        "en": "What can I do if my council or housing association wants to evict me?",
        "pl": "Co mogę zrobić, jeśli lokalny urząd lub stowarzyszenie mieszkaniowe chce mnie eksmitować?",
        "ro": "Ce pot face dacă consiliul local sau asociația de locuințe vrea să mă evacueze?",
        "ar": "ماذا يمكنني أن أفعل إذا أراد المجلس المحلي أو جمعية الإسكان طردي؟",
        "ur": "اگر کونسل یا ہاؤسنگ ایسوسی ایشن مجھے گھر سے نکالنا چاہے تو میں کیا کر سکتا ہوں؟"}),

    "M01": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "change your current GP surgery online", {
        "en": "How do I register with a GP surgery?",
        "pl": "Jak zarejestrować się u lekarza rodzinnego?",
        "ro": "Cum mă înregistrez la un medic de familie?",
        "ar": "كيف أسجل لدى طبيب عام؟",
        "ur": "میں جی پی کے پاس رجسٹریشن کیسے کرواؤں؟"}),
    "M02": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "do not need ID, proof of address", {
        "en": "Do I need proof of address or immigration status to register with a GP?",
        "pl": "Czy potrzebuję potwierdzenia adresu lub statusu imigracyjnego, aby zarejestrować się u lekarza rodzinnego?",
        "ro": "Am nevoie de dovada adresei sau a statutului de imigrant ca să mă înregistrez la medicul de familie?",
        "ar": "هل أحتاج إلى إثبات عنوان أو إثبات وضع الهجرة للتسجيل لدى طبيب عام؟",
        "ur": "کیا جی پی کے پاس رجسٹر ہونے کے لیے مجھے پتے یا امیگریشن اسٹیٹس کا ثبوت چاہیے؟"}),
    "M03": ("health", N + "urgent-and-emergency-care-services/when-to-use-111/", "if you think you need medical help right now", {
        "en": "When should I use NHS 111?",
        "pl": "Kiedy powinienem skorzystać z NHS 111?",
        "ro": "Când ar trebui să folosesc NHS 111?",
        "ar": "متى يجب أن أستخدم خدمة NHS 111؟",
        "ur": "مجھے NHS 111 کب استعمال کرنا چاہیے؟"}),
    "M04": ("health", N + "urgent-and-emergency-care-services/when-to-go-to-ae/", "serious injuries and life-threatening emergencies", {
        "en": "When should I go to A&E?",
        "pl": "Kiedy powinienem iść na szpitalny oddział ratunkowy?",
        "ro": "Când ar trebui să merg la urgențe?",
        "ar": "متى يجب أن أذهب إلى قسم الطوارئ؟",
        "ur": "مجھے ہسپتال کے ایمرجنسی شعبے میں کب جانا چاہیے؟"}),
    "M05": ("health", N + "urgent-and-emergency-care-services/when-to-call-999/", "999 is for life-threatening emergencies", {
        "en": "When should I call 999?",
        "pl": "Kiedy powinienem dzwonić pod numer 999?",
        "ro": "Când ar trebui să sun la 999?",
        "ar": "متى يجب أن أتصل بالرقم 999؟",
        "ur": "مجھے 999 پر کب کال کرنی چاہیے؟"}),
    "M06": ("health", N + "dentists/how-to-find-an-nhs-dentist/", "Ask for an NHS appointment", {
        "en": "How can I find an NHS dentist?",
        "pl": "Jak znaleźć dentystę NHS?",
        "ro": "Cum găsesc un dentist NHS?",
        "ar": "كيف أجد طبيب أسنان تابعًا لـ NHS؟",
        "ur": "میں NHS کا ڈینٹسٹ کیسے تلاش کروں؟"}),
    "M07": ("health", N + "dentists/how-much-nhs-dental-treatment-costs/", "£27.90", {
        "en": "How much does a basic NHS dental check-up cost?",
        "pl": "Ile kosztuje podstawowy przegląd u dentysty w NHS?",
        "ro": "Cât costă un control stomatologic de bază la NHS?",
        "ar": "كم يكلف فحص الأسنان الأساسي في NHS؟",
        "ur": "NHS میں دانتوں کے بنیادی معائنے پر کتنا خرچ آتا ہے؟"}),
    "M08": ("health", N + "pharmacies/how-pharmacies-can-help/", "This is called Pharmacy First", {
        "en": "Can a pharmacy give me prescription medicine without seeing a GP?",
        "pl": "Czy apteka może wydać mi lek na receptę bez wizyty u lekarza?",
        "ro": "Poate o farmacie să-mi dea medicamente cu rețetă fără să merg la medic?",
        "ar": "هل يمكن للصيدلية أن تعطيني دواءً يحتاج إلى وصفة طبية دون أن أرى الطبيب؟",
        "ur": "کیا فارمیسی ڈاکٹر کو دکھائے بغیر مجھے نسخے والی دوا دے سکتی ہے؟"}),
    "M09": ("health", N + "help-with-health-costs/nhs-low-income-scheme-lis/", "HC2 certificate", {
        "en": "Can I get help with health costs if I have a low income?",
        "pl": "Czy mogę dostać pomoc w pokryciu kosztów leczenia, jeśli mam niskie dochody?",
        "ro": "Pot primi ajutor pentru costurile medicale dacă am venituri mici?",
        "ar": "هل يمكنني الحصول على مساعدة في التكاليف الصحية إذا كان دخلي منخفضًا؟",
        "ur": "کیا کم آمدنی کی صورت میں مجھے علاج کے اخراجات میں مدد مل سکتی ہے؟"}),
    "M10": ("health", G + "healthcare-immigration-application", "£776 per year for students", {
        "en": "How much is the immigration health surcharge for students?",
        "pl": "Ile wynosi dopłata zdrowotna dla imigrantów w przypadku studentów?",
        "ro": "Cât costă taxa de sănătate pentru imigranți în cazul studenților?",
        "ar": "كم تبلغ رسوم الرعاية الصحية للمهاجرين بالنسبة للطلاب؟",
        "ur": "طلبہ کے لیے امیگریشن ہیلتھ سرچارج کتنا ہے؟"}),
}


def main():
    pages = {}
    for line in open(ROOT / "data/processed/pages.jsonl"):
        r = json.loads(line)
        pages[r["url"]] = " ".join(s["text"] for s in r["sections"]).lower()
    bad = [(q, url) for q, (_, url, ev, _) in Q.items() if url not in pages or ev.lower() not in pages[url]]
    if bad:
        sys.exit(f"evidence not found on gold page: {bad}")
    out = ROOT / "eval/queries.csv"
    with out.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["qid", "domain", "lang", "question", "gold_url", "evidence", "translation_status"])
        for qid, (domain, url, ev, qs) in Q.items():
            for lang, text in qs.items():
                w.writerow([qid, domain, lang, text, url, ev, "source" if lang == "en" else "unverified"])
    print(f"{len(Q)} questions x {len(next(iter(Q.values()))[3])} languages -> {out.relative_to(ROOT)}; "
          f"all evidence phrases found verbatim on their gold page")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval/build_unanswerable.py
"""Questions the assistant should refuse: the answer is not in the 38-page corpus.

Each topic was checked against data/processed/chunks.csv (11 Nov 2026). Two are deliberately close to
the corpus: "passport" and "birth certificate" appear only as examples of ID documents, so the search
finds related-looking passages that do not answer the question. U05 asks for medical advice, which is
out of scope. Non-English versions were written by the author (translation_status = unverified, T22).

Output: eval/unanswerable.csv (10 questions x 5 languages = 50 rows)
"""
import csv
import pathlib

ROOT = pathlib.Path(__file__).resolve().parents[1]

QUESTIONS = {
    "U01": {"en": "How do I renew my UK passport?",
            "pl": "Jak odnowić brytyjski paszport?",
            "ro": "Cum îmi reînnoiesc pașaportul britanic?",
            "ar": "كيف أجدد جواز سفري البريطاني؟",
            "ur": "میں اپنا برطانوی پاسپورٹ کیسے تجدید کرواؤں؟"},
    "U02": {"en": "What is the minimum wage for someone aged 21?",
            "pl": "Ile wynosi płaca minimalna dla osoby w wieku 21 lat?",
            "ro": "Care este salariul minim pentru o persoană de 21 de ani?",
            "ar": "ما هو الحد الأدنى للأجور لشخص عمره 21 عامًا؟",
            "ur": "21 سال کے شخص کے لیے کم از کم اجرت کتنی ہے؟"},
    "U03": {"en": "How do I apply for British citizenship?",
            "pl": "Jak ubiegać się o brytyjskie obywatelstwo?",
            "ro": "Cum pot aplica pentru cetățenia britanică?",
            "ar": "كيف أتقدم بطلب للحصول على الجنسية البريطانية؟",
            "ur": "میں برطانوی شہریت کے لیے درخواست کیسے دوں؟"},
    "U04": {"en": "How do I register my baby's birth?",
            "pl": "Jak zarejestrować urodzenie mojego dziecka?",
            "ro": "Cum înregistrez nașterea copilului meu?",
            "ar": "كيف أسجل ولادة طفلي؟",
            "ur": "میں اپنے بچے کی پیدائش کا اندراج کیسے کرواؤں؟"},
    "U05": {"en": "What is the best treatment for back pain?",
            "pl": "Jakie jest najlepsze leczenie bólu pleców?",
            "ro": "Care este cel mai bun tratament pentru durerea de spate?",
            "ar": "ما هو أفضل علاج لآلام الظهر؟",
            "ur": "کمر درد کا بہترین علاج کیا ہے؟"},
    "U06": {"en": "How do I get a mortgage to buy my first home?",
            "pl": "Jak dostać kredyt hipoteczny na zakup pierwszego mieszkania?",
            "ro": "Cum obțin un credit ipotecar pentru a-mi cumpăra prima locuință?",
            "ar": "كيف أحصل على قرض عقاري لشراء أول منزل لي؟",
            "ur": "میں اپنا پہلا گھر خریدنے کے لیے مارگیج کیسے حاصل کروں؟"},
    "U07": {"en": "How do I register to vote?",
            "pl": "Jak zarejestrować się do głosowania?",
            "ro": "Cum mă înscriu pentru a vota?",
            "ar": "كيف أسجل للتصويت؟",
            "ur": "میں ووٹ ڈالنے کے لیے رجسٹریشن کیسے کرواؤں؟"},
    "U08": {"en": "Do I need a TV licence to watch TV online?",
            "pl": "Czy potrzebuję licencji telewizyjnej, żeby oglądać telewizję w internecie?",
            "ro": "Am nevoie de licență TV ca să mă uit la televizor online?",
            "ar": "هل أحتاج إلى ترخيص تلفزيون لمشاهدة التلفزيون عبر الإنترنت؟",
            "ur": "کیا مجھے آن لائن ٹی وی دیکھنے کے لیے ٹی وی لائسنس کی ضرورت ہے؟"},
    "U09": {"en": "How much does a driving test cost?",
            "pl": "Ile kosztuje egzamin na prawo jazdy?",
            "ro": "Cât costă examenul pentru permisul de conducere?",
            "ar": "كم تكلفة اختبار القيادة؟",
            "ur": "ڈرائیونگ ٹیسٹ کی فیس کتنی ہے؟"},
    "U10": {"en": "Can I get a free bus pass because I am over 60?",
            "pl": "Czy mogę dostać darmowy bilet autobusowy, bo mam ponad 60 lat?",
            "ro": "Pot primi un abonament gratuit de autobuz pentru că am peste 60 de ani?",
            "ar": "هل يمكنني الحصول على تذكرة حافلة مجانية لأن عمري أكثر من 60 عامًا؟",
            "ur": "کیا مجھے 60 سال سے زیادہ عمر ہونے کی وجہ سے مفت بس پاس مل سکتا ہے؟"},
}

if __name__ == "__main__":
    out = ROOT / "eval/unanswerable.csv"
    with open(out, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["qid", "lang", "question", "translation_status"])
        for qid, by_lang in QUESTIONS.items():
            for lang, q in by_lang.items():
                w.writerow([qid, lang, q, "source" if lang == "en" else "unverified"])
    print("wrote", out, 5 * len(QUESTIONS), "rows")


In [ ]:
%%writefile eval/build_gold_set.py
"""Week 7 gold set: 100 question topics with reference answers, split into calibration and held-out.

  calibration (40 topics) - the week 4-6 questions: 30 answerable (eval/queries.csv) and 10 unanswerable
                            (eval/unanswerable.csv). The week 6 refusal threshold was set on these.
  held-out    (60 topics) - new in week 7: 50 answerable (S11-S26, H11-H27, M11-M27) and 10 unanswerable
                            (U11-U20). Never used to tune anything; used to check the refusal rules and,
                            in week 9, for the grounded vs ungrounded comparison.

Every answerable topic has
  gold_url         - the page that answers it
  evidence         - a phrase copied verbatim from that page (checked below; the build stops if one is missing)
  reference_answer - a short answer written by the author from that page's text, used by the scoring rubric
Unanswerable topics were checked against the corpus (11-16 Nov 2026): the topic is absent, or only mentioned
in passing without an answer (U12 National Insurance number, U14 student loan, U17 Carer's Allowance).

Non-English questions were written by the author (translation_status = unverified, task T22).
Outputs: eval/heldout_queries.csv, eval/heldout_unanswerable.csv, eval/gold_set.csv
"""
import csv
import json
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
G = "https://www.gov.uk/"
N = "https://www.nhs.uk/nhs-services/"
GP = N + "gps/how-to-register-with-a-gp-surgery/"
GPA = N + "gps/gp-appointments-and-bookings/"
U111 = N + "urgent-and-emergency-care-services/when-to-use-111/"
AE = N + "urgent-and-emergency-care-services/when-to-go-to-ae/"
C999 = N + "urgent-and-emergency-care-services/when-to-call-999/"
DENT = N + "dentists/how-much-nhs-dental-treatment-costs/"
PHARM = N + "pharmacies/how-pharmacies-can-help/"
LIS = N + "help-with-health-costs/nhs-low-income-scheme-lis/"
ABROAD = (N + "visiting-or-moving-to-england/"
          "how-to-access-nhs-services-in-england-if-you-are-visiting-from-abroad/")
VACC = "https://www.nhs.uk/vaccinations/nhs-vaccinations-and-when-to-have-them/"
PREG = "https://www.nhs.uk/pregnancy/finding-out/what-to-do-when-you-find-out-you-are-pregnant/"

# Reference answers for the 30 calibration questions (questions and evidence are in eval/build_queries.py).
CALIBRATION_REFERENCE = {
    "S01": "Applications open in September and close on 15 January. You apply through your local council, "
           "a year before your child can start school.",
    "S02": "You must apply by 31 October. Applications usually open on 1 September.",
    "S03": "Councils send primary school offers on 16 April (the next working day if that is a weekend or "
           "bank holiday).",
    "S04": "The admission authority must allow you at least 20 school days to appeal from when it sends "
           "the decision letter.",
    "S05": "Yes. You can be fined or face legal action. The council can fine each parent £80, rising to £160 "
           "if not paid within 21 days.",
    "S06": "Yes, if your household income is £7,400 a year or less (after tax, not counting benefits). From "
           "the 2026 to 2027 school year, any family on Universal Credit qualifies, whatever the income.",
    "S07": "Yes, if it is the nearest suitable school and it is more than 2 miles away (child under 8) or "
           "more than 3 miles away (child 8 or over), or there is no safe walking route, or the child cannot "
           "walk because of SEND or a mobility problem.",
    "S08": "On the last Friday in June if they will be 16 by the end of the summer holidays. Until 18 they "
           "must stay in education, start an apprenticeship, or work or volunteer 20 hours a week with "
           "part-time education or training.",
    "S09": "SEN support in school (for example speech therapy), or an education, health and care (EHC) plan "
           "for more complex needs, up to age 25. Talk to the school's SENCO.",
    "S10": "Talk to the school first. If that does not solve it, use the school's formal complaints "
           "procedure; after that you can complain to the Department for Education or Ofsted.",
    "H01": "No. From 1 May 2026 the 'no fault' section 21 process ended. The landlord must give a legal "
           "reason (a ground for possession) and the correct notice, usually 4 months.",
    "H02": "Once a year, and not in the first 12 months of a new tenancy. The landlord must use Form 4A and "
           "give at least 2 months' notice. You can challenge an increase above market rent.",
    "H03": "Yes, if you have an assured periodic tenancy and the deposit was taken on or after 6 April 2007. "
           "It must go into a government-approved scheme within 30 days.",
    "H04": "Housing Benefit can help pay rent if you are unemployed, on a low income or claiming benefits, but "
           "new claims are only possible at State Pension age or in supported, sheltered or temporary housing. "
           "Most people claim Universal Credit instead.",
    "H05": "Apply through your local council. You usually join a waiting list, and councils decide who gets "
           "housing using points or bands based on need.",
    "H06": "Yes. Contact the local council for help and advice if you are homeless or about to lose your home.",
    "H07": "Yes. The landlord must keep the property safe and in a good state of repair, and is always "
           "responsible for the structure, heating, hot water, gas, wiring and sanitary fittings.",
    "H08": "At least 24 hours' notice, at a reasonable time of day, unless it is an emergency.",
    "H09": "Yes, 25% off the bill if you live on your own. You need to apply.",
    "H10": "You may be able to stop or delay it. Get advice from Citizens Advice or Shelter and check if you "
           "can get legal aid. The landlord must give written notice and needs a court order.",
    "M01": "Online through the NHS website (or the NHS App to change surgery), or with a paper form or the "
           "surgery's own website. It is free.",
    "M02": "No. You only need basic information such as your name, date of birth and address. You do not "
           "need ID, proof of address or proof of immigration status.",
    "M03": "When you think you need medical help right now but it is not life-threatening, or you are not "
           "sure where to go. Use 111 online (age 5 and over) or call 111.",
    "M04": "For serious injuries and life-threatening emergencies.",
    "M05": "For life-threatening emergencies, such as serious road accidents, strokes and heart attacks.",
    "M06": "Contact any NHS dentist and ask for an NHS appointment. If you cannot find one, contact your "
           "local integrated care board (ICB).",
    "M07": "A Band 1 check-up (examination, X-rays if needed, scaling) costs £27.90.",
    "M08": "Yes, through Pharmacy First, for some conditions such as sore throat, earache, sinusitis, "
           "impetigo, infected insect bites, UTIs and shingles (age limits apply).",
    "M09": "Yes, through the NHS Low Income Scheme if your savings are under £16,000 (£23,250 in a care "
           "home). It gives full help (HC2) or partial help (HC3).",
    "M10": "£776 per year for students and their dependants.",
}

# Held-out answerable: qid -> (domain, gold_url, evidence, reference_answer, {lang: question})
HELDOUT = {
    # ---------- schools ----------
    "S11": ("schools", G + "schools-admissions", "start reception full-time in september after their fourth birthday",
            "Most children start reception full-time in the September after their fourth birthday.", {
        "en": "At what age does my child start school?",
        "pl": "W jakim wieku moje dziecko zaczyna szkołę?",
        "ro": "La ce vârstă începe copilul meu școala?",
        "ar": "في أي عمر يبدأ طفلي المدرسة؟",
        "ur": "میرا بچہ کس عمر میں اسکول شروع کرتا ہے؟"}),
    "S12": ("schools", G + "schools-admissions", "class size is limited to 30",
            "In reception, year 1 and year 2 classes are limited to 30 children.", {
        "en": "How many children can be in an infant class?",
        "pl": "Ile dzieci może być w klasie dla najmłodszych uczniów (infant class)?",
        "ro": "Câți copii pot fi într-o clasă pentru cei mici (infant class)?",
        "ar": "كم عدد الأطفال المسموح به في صف الأطفال الصغار (infant class)؟",
        "ur": "چھوٹے بچوں کی کلاس (infant class) میں کتنے بچے ہو سکتے ہیں؟"}),
    "S13": ("schools", G + "schools-admissions", "the local council must provide a place at another school",
            "Yes. If your child is not offered any of the schools you applied for, the council must provide a "
            "place at another school, usually the nearest one with places.", {
        "en": "What happens if my child does not get a place at any school I applied for?",
        "pl": "Co się stanie, jeśli moje dziecko nie dostanie miejsca w żadnej szkole, do której złożyłem wniosek?",
        "ro": "Ce se întâmplă dacă copilul meu nu primește loc la niciuna dintre școlile la care am aplicat?",
        "ar": "ماذا يحدث إذا لم يحصل طفلي على مقعد في أي مدرسة تقدمت إليها؟",
        "ur": "اگر میرے بچے کو کسی بھی اسکول میں جگہ نہ ملے جہاں میں نے درخواست دی تھی تو کیا ہوگا؟"}),
    "S14": ("schools", G + "schools-admissions", "for at least the first term of each school year",
            "The admission authority must keep a waiting list open for at least the first term of each school "
            "year.", {
        "en": "How long does a school have to keep its waiting list open?",
        "pl": "Jak długo szkoła musi prowadzić listę oczekujących?",
        "ro": "Cât timp trebuie o școală să păstreze deschisă lista de așteptare?",
        "ar": "إلى متى يجب على المدرسة أن تُبقي قائمة الانتظار مفتوحة؟",
        "ur": "اسکول کو اپنی ویٹنگ لسٹ کتنے عرصے تک کھلی رکھنی ہوتی ہے؟"}),
    "S15": ("schools", G + "school-attendance-absence", "if you do not pay the fine in 28 days",
            "If you do not pay the fine in 28 days you may be taken to court, where you could be fined up to "
            "£2,500, get a community order or a jail sentence of up to 3 months.", {
        "en": "What happens if I do not pay a school absence fine?",
        "pl": "Co się stanie, jeśli nie zapłacę grzywny za nieobecność dziecka w szkole?",
        "ro": "Ce se întâmplă dacă nu plătesc amenda pentru absența copilului de la școală?",
        "ar": "ماذا يحدث إذا لم أدفع غرامة غياب طفلي عن المدرسة؟",
        "ur": "اگر میں اسکول سے غیر حاضری کا جرمانہ ادا نہ کروں تو کیا ہوگا؟"}),
    "S16": ("schools", G + "school-attendance-absence", "you have advance permission from the school",
            "Only if they are too ill to go in, or you have advance permission from the school.", {
        "en": "When is my child allowed to miss school?",
        "pl": "Kiedy moje dziecko może opuścić zajęcia w szkole?",
        "ro": "Când are voie copilul meu să lipsească de la școală?",
        "ar": "متى يُسمح لطفلي بالتغيب عن المدرسة؟",
        "ur": "میرے بچے کو کب اسکول سے غیر حاضر رہنے کی اجازت ہے؟"}),
    "S17": ("schools", G + "apply-free-school-meals", "reception class year 1 year 2",
            "Yes. Children in reception, year 1 and year 2 at a government-funded school can get free school "
            "meals.", {
        "en": "Can my child in reception get free school meals?",
        "pl": "Czy moje dziecko w zerówce (reception) może dostać darmowe posiłki w szkole?",
        "ro": "Poate copilul meu din clasa reception să primească mese gratuite la școală?",
        "ar": "هل يمكن لطفلي في صف الروضة (reception) الحصول على وجبات مدرسية مجانية؟",
        "ur": "کیا ریسیپشن کلاس میں میرے بچے کو اسکول میں مفت کھانا مل سکتا ہے؟"}),
    "S18": ("schools", G + "free-school-transport", "more than 3 miles away and the child is 8 or over",
            "For a child aged 8 or over, free transport applies if the nearest suitable school is more than 3 "
            "miles away (other conditions can also qualify).", {
        "en": "My child is 9. How far away must the school be to get free school transport?",
        "pl": "Moje dziecko ma 9 lat. Jak daleko musi być szkoła, żeby przysługiwał darmowy transport?",
        "ro": "Copilul meu are 9 ani. Cât de departe trebuie să fie școala ca să primească transport gratuit?",
        "ar": "عمر طفلي 9 سنوات. ما المسافة التي يجب أن تبعدها المدرسة للحصول على نقل مدرسي مجاني؟",
        "ur": "میرا بچہ 9 سال کا ہے۔ مفت اسکول ٹرانسپورٹ کے لیے اسکول کتنا دور ہونا چاہیے؟"}),
    "S19": ("schools", G + "school-uniform", "more than 3 items of branded school uniform",
            "From September 2026 schools should not require more than 3 branded items (4 for secondary and "
            "middle schools if one is a tie).", {
        "en": "How many branded uniform items can a school make me buy?",
        "pl": "Ile elementów mundurka z logo szkoły szkoła może kazać mi kupić?",
        "ro": "Câte articole de uniformă cu sigla școlii mă poate obliga școala să cumpăr?",
        "ar": "كم عدد قطع الزي المدرسي التي تحمل شعار المدرسة التي يمكن للمدرسة أن تُلزمني بشرائها؟",
        "ur": "اسکول مجھے اسکول کے لوگو والی یونیفارم کی کتنی اشیاء خریدنے پر مجبور کر سکتا ہے؟"}),
    "S20": ("schools", G + "children-with-special-educational-needs", "within 16 weeks",
            "The local authority must tell you within 16 weeks whether an EHC plan will be made; the final "
            "plan is due within 20 weeks of the request.", {
        "en": "How long does the council take to decide on an EHC plan?",
        "pl": "Ile czasu ma samorząd na decyzję w sprawie planu EHC?",
        "ro": "Cât timp are consiliul local ca să decidă asupra unui plan EHC?",
        "ar": "كم من الوقت يستغرق المجلس المحلي لاتخاذ قرار بشأن خطة EHC؟",
        "ur": "کونسل کو EHC پلان کے بارے میں فیصلہ کرنے میں کتنا وقت لگتا ہے؟"}),
    "S21": ("schools", G + "children-with-special-educational-needs",
            "you can ask your local authority to carry out an assessment",
            "Yes. You can ask your local authority to carry out an EHC needs assessment.", {
        "en": "Can I ask the council to assess my child for an EHC plan?",
        "pl": "Czy mogę poprosić samorząd o ocenę potrzeb mojego dziecka w celu przyznania planu EHC?",
        "ro": "Pot cere consiliului local să evalueze copilul meu pentru un plan EHC?",
        "ar": "هل يمكنني أن أطلب من المجلس المحلي تقييم طفلي للحصول على خطة EHC؟",
        "ur": "کیا میں کونسل سے اپنے بچے کا EHC پلان کے لیے جائزہ لینے کی درخواست کر سکتا ہوں؟"}),
    "S22": ("schools", G + "types-of-school", "free schools are funded by the government but are not run by the local authority",
            "Free schools are funded by the government but not run by the local authority. They are "
            "all-ability, can change term lengths and do not have to follow the national curriculum.", {
        "en": "What is a free school?",
        "pl": "Czym jest szkoła typu free school?",
        "ro": "Ce este o școală de tip free school?",
        "ar": "ما هي المدرسة من نوع free school؟",
        "ur": "فری اسکول (free school) کیا ہوتا ہے؟"}),
    "S23": ("schools", G + "types-of-school", "academies do not charge fees",
            "No. Academies are funded directly by the government and do not charge fees.", {
        "en": "Do academies charge school fees?",
        "pl": "Czy akademie (academies) pobierają opłaty za naukę?",
        "ro": "Academiile (academies) percep taxe școlare?",
        "ar": "هل تفرض مدارس الأكاديمية (academies) رسومًا دراسية؟",
        "ur": "کیا اکیڈمی اسکول فیس لیتے ہیں؟"}),
    "S24": ("schools", G + "school-term-holiday-dates", "holiday dates on your local council",
            "Term, half term and holiday dates are on your local council's website.", {
        "en": "Where can I find my child's school holiday dates?",
        "pl": "Gdzie znajdę daty ferii i przerw szkolnych mojego dziecka?",
        "ro": "Unde găsesc datele vacanțelor școlare ale copilului meu?",
        "ar": "أين أجد مواعيد العطل المدرسية لطفلي؟",
        "ur": "مجھے اپنے بچے کی اسکول کی چھٹیوں کی تاریخیں کہاں ملیں گی؟"}),
    "S25": ("schools", G + "know-when-you-can-leave-school", "stay in full-time education, for example at a college",
            "Until 18 they must stay in full-time education, start an apprenticeship, or spend 20 hours or "
            "more a week working or volunteering while in part-time education or training.", {
        "en": "My child left school at 16. What must they do until they are 18?",
        "pl": "Moje dziecko skończyło szkołę w wieku 16 lat. Co musi robić do 18. roku życia?",
        "ro": "Copilul meu a terminat școala la 16 ani. Ce trebuie să facă până la 18 ani?",
        "ar": "ترك طفلي المدرسة في سن 16. ما الذي يجب عليه فعله حتى يبلغ 18 عامًا؟",
        "ur": "میرے بچے نے 16 سال کی عمر میں اسکول چھوڑ دیا۔ 18 سال کی عمر تک اسے کیا کرنا ہوگا؟"}),
    "S26": ("schools", G + "complain-about-school", "within 6 months of the discrimination taking place",
            "You must send a disability discrimination complaint to the SEND tribunal within 6 months of the "
            "discrimination. It is free.", {
        "en": "How long do I have to complain to the SEND tribunal about disability discrimination at school?",
        "pl": "Ile mam czasu na złożenie skargi do trybunału SEND w sprawie dyskryminacji z powodu niepełnosprawności w szkole?",
        "ro": "Cât timp am ca să depun o plângere la tribunalul SEND pentru discriminare pe motiv de dizabilitate la școală?",
        "ar": "ما المدة المتاحة لي لتقديم شكوى إلى محكمة SEND بشأن التمييز بسبب الإعاقة في المدرسة؟",
        "ur": "اسکول میں معذوری کی بنیاد پر امتیاز کی شکایت SEND ٹریبونل میں کرنے کے لیے میرے پاس کتنا وقت ہے؟"}),
    # ---------- housing ----------
    "H11": ("housing", G + "private-renting", "rent for the year is less than £50,000",
            "Up to 5 weeks' rent if the yearly rent is under £50,000, or 6 weeks' rent if it is £50,000 or "
            "more. A holding deposit can be up to one week's rent.", {
        "en": "What is the most a landlord can ask for as a deposit?",
        "pl": "Jaką maksymalną kaucję może pobrać wynajmujący?",
        "ro": "Care este garanția maximă pe care o poate cere proprietarul?",
        "ar": "ما هو الحد الأقصى لمبلغ التأمين الذي يمكن أن يطلبه المالك؟",
        "ur": "مالک مکان زیادہ سے زیادہ کتنا ڈپازٹ مانگ سکتا ہے؟"}),
    "H12": ("housing", G + "tenancy-deposit-protection", "within 30 days of getting it",
            "Your landlord or letting agent must put the deposit in a government-approved scheme within 30 days "
            "of getting it.", {
        "en": "How quickly must my landlord put my deposit in a protection scheme?",
        "pl": "W jakim terminie wynajmujący musi umieścić moją kaucję w systemie ochrony depozytów?",
        "ro": "În cât timp trebuie proprietarul să pună garanția mea într-o schemă de protecție?",
        "ar": "خلال كم من الوقت يجب على المالك إيداع مبلغ التأمين في نظام حماية؟",
        "ur": "مالک مکان کو میرا ڈپازٹ کتنے دنوں میں تحفظ کی اسکیم میں رکھنا ہوتا ہے؟"}),
    "H13": ("housing", G + "tenancy-deposit-protection", "within 10 days of you both agreeing",
            "The landlord must return the deposit within 10 days of you both agreeing how much you will get back.", {
        "en": "How long does my landlord have to give my deposit back when I move out?",
        "pl": "Ile czasu ma wynajmujący na zwrot kaucji, gdy się wyprowadzam?",
        "ro": "Cât timp are proprietarul ca să-mi returneze garanția când mă mut?",
        "ar": "كم من الوقت لدى المالك لإعادة مبلغ التأمين عندما أغادر السكن؟",
        "ur": "جب میں گھر چھوڑوں تو مالک مکان کے پاس میرا ڈپازٹ واپس کرنے کے لیے کتنا وقت ہے؟"}),
    "H14": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "end it by giving 2 months",
            "You can end an assured periodic tenancy by giving 2 months' notice in writing, and you keep paying "
            "rent during the notice period.", {
        "en": "How much notice do I need to give to leave my rented home?",
        "pl": "Z jakim wyprzedzeniem muszę wypowiedzieć umowę najmu, żeby się wyprowadzić?",
        "ro": "Cu cât timp înainte trebuie să anunț că mă mut din locuința închiriată?",
        "ar": "ما مدة الإشعار التي يجب أن أقدمها لمغادرة المنزل المستأجر؟",
        "ur": "کرائے کا گھر چھوڑنے کے لیے مجھے کتنا پہلے نوٹس دینا ہوگا؟"}),
    "H15": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "a maximum of 1 month",
            "Nothing before you sign the tenancy agreement; after signing, a maximum of 1 month's rent in advance.", {
        "en": "How much rent can a landlord ask me to pay in advance?",
        "pl": "Ile czynszu z góry może żądać wynajmujący?",
        "ro": "Cât din chirie îmi poate cere proprietarul să plătesc în avans?",
        "ar": "كم من الإيجار يمكن للمالك أن يطلب مني دفعه مقدمًا؟",
        "ur": "مالک مکان مجھ سے کتنا کرایہ پیشگی ادا کرنے کا کہہ سکتا ہے؟"}),
    "H16": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "you can ask to keep a pet in the property",
            "Yes, you can ask. The landlord must consider the request and should give a reason if they refuse.", {
        "en": "Can I keep a pet in my rented home?",
        "pl": "Czy mogę trzymać zwierzę w wynajmowanym mieszkaniu?",
        "ro": "Pot ține un animal de companie în locuința închiriată?",
        "ar": "هل يمكنني تربية حيوان أليف في المنزل المستأجر؟",
        "ur": "کیا میں کرائے کے گھر میں پالتو جانور رکھ سکتا ہوں؟"}),
    "H17": ("housing", G + "guidance/renters-rights-act-overview-for-tenants",
            "cannot refuse to rent a property to a tenant who is on benefits",
            "No. A landlord cannot refuse to rent to a tenant because they are on benefits or have children.", {
        "en": "Can a landlord refuse to rent to me because I get benefits?",
        "pl": "Czy wynajmujący może odmówić mi wynajmu, bo pobieram zasiłki?",
        "ro": "Poate un proprietar să refuze să-mi închirieze pentru că primesc ajutoare sociale?",
        "ar": "هل يمكن للمالك أن يرفض تأجيري لأنني أحصل على إعانات؟",
        "ur": "کیا مالک مکان مجھے اس وجہ سے کرایہ پر گھر دینے سے انکار کر سکتا ہے کہ میں بینیفٹس لیتا ہوں؟"}),
    "H18": ("housing", G + "private-renting", "environmental health department at your local council",
            "Contact the environmental health department at your local council. It must act if the problems "
            "could harm you.", {
        "en": "My landlord will not do the repairs. Who can help me?",
        "pl": "Wynajmujący nie chce zrobić napraw. Kto może mi pomóc?",
        "ro": "Proprietarul nu vrea să facă reparațiile. Cine mă poate ajuta?",
        "ar": "المالك لا يقوم بالإصلاحات. من يمكنه مساعدتي؟",
        "ur": "میرا مالک مکان مرمت نہیں کر رہا۔ میری مدد کون کر سکتا ہے؟"}),
    "H19": ("housing", G + "private-renting", "annual gas safety check",
            "Yes. A registered engineer must do an annual gas safety check, and the landlord must give you a "
            "copy before you move in or within 28 days of the check.", {
        "en": "Does my landlord have to do a gas safety check?",
        "pl": "Czy wynajmujący musi przeprowadzać kontrolę bezpieczeństwa instalacji gazowej?",
        "ro": "Este proprietarul obligat să facă o verificare a siguranței instalației de gaz?",
        "ar": "هل يجب على المالك إجراء فحص لسلامة الغاز؟",
        "ur": "کیا مالک مکان کو گیس کی حفاظتی جانچ کروانی ضروری ہے؟"}),
    "H20": ("housing", G + "private-renting-evictions", "fine of up to £40,000",
            "Local councils can prosecute landlords or give a fine of up to £40,000 for harassment or illegal "
            "eviction.", {
        "en": "How much can a landlord be fined for illegally evicting a tenant?",
        "pl": "Jaką grzywnę może dostać wynajmujący za nielegalną eksmisję najemcy?",
        "ro": "Ce amendă poate primi un proprietar pentru evacuarea ilegală a unui chiriaș?",
        "ar": "ما قيمة الغرامة التي يمكن فرضها على المالك بسبب الإخلاء غير القانوني للمستأجر؟",
        "ur": "کرایہ دار کو غیر قانونی طور پر نکالنے پر مالک مکان کو کتنا جرمانہ ہو سکتا ہے؟"}),
    "H21": ("housing", G + "private-renting-evictions", "withholding keys",
            "Harassment can include stopping services like electricity, withholding keys, refusing repairs, "
            "threats, threatening to change the locks and opening or taking post.", {
        "en": "What counts as harassment by a landlord?",
        "pl": "Co jest uważane za nękanie ze strony wynajmującego?",
        "ro": "Ce se consideră hărțuire din partea proprietarului?",
        "ar": "ما الذي يُعد تحرشًا أو مضايقة من قبل المالك؟",
        "ur": "مالک مکان کی طرف سے ہراسانی میں کیا شامل ہے؟"}),
    "H22": ("housing", G + "housing-benefit", "reduced by 14%",
            "In council or social housing, Housing Benefit is reduced by 14% of the eligible rent for 1 spare "
            "bedroom and 25% for 2 or more.", {
        "en": "Will my Housing Benefit go down if I have a spare bedroom?",
        "pl": "Czy mój dodatek mieszkaniowy zostanie obniżony, jeśli mam wolną sypialnię?",
        "ro": "Îmi va scădea Housing Benefit dacă am un dormitor liber?",
        "ar": "هل ستنخفض إعانة السكن (Housing Benefit) إذا كان لدي غرفة نوم إضافية؟",
        "ur": "اگر میرے پاس ایک فالتو بیڈروم ہو تو کیا میرا ہاؤسنگ بینیفٹ کم ہو جائے گا؟"}),
    "H23": ("housing", G + "universal-credit", "around 5 weeks to get your first payment",
            "It usually takes around 5 weeks. You can apply for an advance if you need money while you wait.", {
        "en": "How long does it take to get the first Universal Credit payment?",
        "pl": "Ile czasu trzeba czekać na pierwszą wypłatę Universal Credit?",
        "ro": "Cât durează până primesc prima plată Universal Credit?",
        "ar": "كم من الوقت يستغرق الحصول على أول دفعة من Universal Credit؟",
        "ur": "یونیورسل کریڈٹ کی پہلی ادائیگی ملنے میں کتنا وقت لگتا ہے؟"}),
    "H24": ("housing", G + "universal-credit", "£16,000 or less in money, savings and investments",
            "You must have £16,000 or less in money, savings and investments. Over £6,000 reduces the payment.", {
        "en": "How much savings can I have and still claim Universal Credit?",
        "pl": "Ile oszczędności mogę mieć, żeby nadal ubiegać się o Universal Credit?",
        "ro": "Câte economii pot avea ca să pot solicita în continuare Universal Credit?",
        "ar": "ما مقدار المدخرات التي يمكنني امتلاكها مع الاستمرار في المطالبة بـ Universal Credit؟",
        "ur": "یونیورسل کریڈٹ کے لیے درخواست دیتے ہوئے میرے پاس زیادہ سے زیادہ کتنی بچت ہو سکتی ہے؟"}),
    "H25": ("housing", G + "universal-credit", "£303.94 a month for each child",
            "£303.94 a month for each child who lives with you, with an extra £47.94 for a first child born "
            "before 6 April 2017.", {
        "en": "How much extra Universal Credit do I get for each child?",
        "pl": "Ile dodatkowych pieniędzy z Universal Credit dostaję na każde dziecko?",
        "ro": "Câți bani în plus primesc din Universal Credit pentru fiecare copil?",
        "ar": "كم أحصل من مبلغ إضافي في Universal Credit عن كل طفل؟",
        "ur": "یونیورسل کریڈٹ میں ہر بچے کے لیے مجھے کتنی اضافی رقم ملتی ہے؟"}),
    "H26": ("housing", G + "council-tax", "a full-time student do not have to pay council tax",
            "Households where everyone is a full-time student do not have to pay Council Tax. If you get a "
            "bill, apply for an exemption.", {
        "en": "Do students have to pay Council Tax?",
        "pl": "Czy studenci muszą płacić Council Tax?",
        "ro": "Studenții trebuie să plătească Council Tax?",
        "ar": "هل يجب على الطلاب دفع ضريبة المجلس (Council Tax)؟",
        "ur": "کیا طلبہ کو کونسل ٹیکس ادا کرنا پڑتا ہے؟"}),
    "H27": ("housing", G + "council-tax", "over 12 months instead of 10",
            "Yes. Contact your council straight away; it can spread payments over 12 months instead of 10.", {
        "en": "I am struggling to pay my Council Tax. Can I pay it over 12 months?",
        "pl": "Mam problem z płaceniem Council Tax. Czy mogę rozłożyć płatności na 12 miesięcy?",
        "ro": "Îmi este greu să plătesc Council Tax. Pot să-l plătesc în 12 luni?",
        "ar": "أواجه صعوبة في دفع ضريبة المجلس. هل يمكنني دفعها على 12 شهرًا؟",
        "ur": "مجھے کونسل ٹیکس ادا کرنے میں مشکل ہو رہی ہے۔ کیا میں اسے 12 مہینوں میں ادا کر سکتا ہوں؟"}),
    # ---------- health ----------
    "M11": ("health", GP, "usually be within 5 days",
            "You should be told when you are registered, usually within 5 days of the surgery getting your "
            "details, but it may take longer.", {
        "en": "How long does it take to be registered with a GP after I apply online?",
        "pl": "Ile trwa rejestracja u lekarza rodzinnego po złożeniu wniosku online?",
        "ro": "Cât durează înregistrarea la medicul de familie după ce aplic online?",
        "ar": "كم من الوقت يستغرق التسجيل لدى الطبيب العام بعد التقديم عبر الإنترنت؟",
        "ur": "آن لائن درخواست دینے کے بعد جی پی کے ساتھ رجسٹریشن میں کتنا وقت لگتا ہے؟"}),
    "M12": ("health", GP, "temporary patient for up to 3 months",
            "Yes. You can register with a GP surgery as a temporary patient for up to 3 months using a "
            "temporary services form.", {
        "en": "Can I see a GP if I am only staying in an area for a few weeks?",
        "pl": "Czy mogę pójść do lekarza rodzinnego, jeśli przebywam w danej okolicy tylko kilka tygodni?",
        "ro": "Pot merge la un medic de familie dacă stau într-o zonă doar câteva săptămâni?",
        "ar": "هل يمكنني رؤية طبيب عام إذا كنت سأبقى في منطقة ما لبضعة أسابيع فقط؟",
        "ur": "اگر میں کسی علاقے میں صرف چند ہفتے رہ رہا ہوں تو کیا میں جی پی سے مل سکتا ہوں؟"}),
    "M13": ("health", GP, "write to you within 14 days to explain why",
            "Yes. If a GP surgery refuses to register you, it must write to you within 14 days to explain why.", {
        "en": "If a GP surgery refuses to register me, do they have to tell me why?",
        "pl": "Jeśli przychodnia odmówi mi rejestracji, czy musi podać powód?",
        "ro": "Dacă un cabinet de medicină de familie refuză să mă înregistreze, trebuie să-mi spună de ce?",
        "ar": "إذا رفضت عيادة الطبيب العام تسجيلي، هل يجب أن تخبرني بالسبب؟",
        "ur": "اگر جی پی سرجری مجھے رجسٹر کرنے سے انکار کرے تو کیا انہیں وجہ بتانی ہوگی؟"}),
    "M14": ("health", GPA, "saturdays between 9am and 5pm",
            "Yes. You can see a healthcare professional on weekday evenings 6.30pm to 8pm and on Saturdays 9am "
            "to 5pm.", {
        "en": "Can I get a GP appointment in the evening or at the weekend?",
        "pl": "Czy mogę umówić się do lekarza rodzinnego wieczorem lub w weekend?",
        "ro": "Pot primi o programare la medicul de familie seara sau în weekend?",
        "ar": "هل يمكنني الحصول على موعد مع الطبيب العام في المساء أو في عطلة نهاية الأسبوع؟",
        "ur": "کیا مجھے شام یا ہفتے کے آخر میں جی پی کی اپائنٹمنٹ مل سکتی ہے؟"}),
    "M15": ("health", U111, "111 online is for people aged 5 and over",
            "No. 111 online is for people aged 5 and over; call 111 for a child under 5.", {
        "en": "Can I use 111 online for my 3-year-old?",
        "pl": "Czy mogę skorzystać z 111 online dla mojego 3-letniego dziecka?",
        "ro": "Pot folosi 111 online pentru copilul meu de 3 ani?",
        "ar": "هل يمكنني استخدام خدمة 111 عبر الإنترنت لطفلي البالغ من العمر 3 سنوات؟",
        "ur": "کیا میں اپنے 3 سالہ بچے کے لیے 111 آن لائن استعمال کر سکتا ہوں؟"}),
    "M16": ("health", U111, "emergency supply of regularly prescribed medicine",
            "Yes. 111 online can send a request to a local pharmacy for an emergency supply of your regular "
            "medicine until you can get your repeat prescription.", {
        "en": "I have run out of my regular medicine. Can 111 help?",
        "pl": "Skończyły mi się leki, które stale przyjmuję. Czy 111 może pomóc?",
        "ro": "Am rămas fără medicamentele pe care le iau regulat. Mă poate ajuta 111?",
        "ar": "نفد دوائي الذي أتناوله بانتظام. هل يمكن لخدمة 111 مساعدتي؟",
        "ur": "میری باقاعدہ دوا ختم ہو گئی ہے۔ کیا 111 مدد کر سکتا ہے؟"}),
    "M17": ("health", AE, "do not drive to a&e",
            "No. Do not drive to A&E. Ask someone to drive you or call 999 for an ambulance.", {
        "en": "Should I drive myself to A&E?",
        "pl": "Czy powinienem sam pojechać samochodem na SOR (A&E)?",
        "ro": "Ar trebui să conduc singur până la urgențe (A&E)?",
        "ar": "هل يجب أن أقود السيارة بنفسي إلى قسم الطوارئ (A&E)؟",
        "ur": "کیا مجھے خود گاڑی چلا کر A&E جانا چاہیے؟"}),
    "M18": ("health", C999, "calling 999 does not always mean an ambulance will be sent",
            "No. Calling 999 does not always mean an ambulance is sent; you may be told to make your own way "
            "to A&E or be seen elsewhere.", {
        "en": "If I call 999, will an ambulance always come?",
        "pl": "Czy jeśli zadzwonię pod 999, zawsze przyjedzie karetka?",
        "ro": "Dacă sun la 999, va veni întotdeauna o ambulanță?",
        "ar": "إذا اتصلت بالرقم 999، هل ستأتي سيارة إسعاف دائمًا؟",
        "ur": "اگر میں 999 پر کال کروں تو کیا ایمبولینس ہمیشہ آئے گی؟"}),
    "M19": ("health", DENT, "within 2 months, you do not have to pay extra",
            "If the further treatment within 2 months is in the same or a lower band, you do not pay extra; a "
            "higher band means paying the higher fee.", {
        "en": "Do I have to pay again if I need more dental treatment a few weeks later?",
        "pl": "Czy muszę płacić ponownie, jeśli po kilku tygodniach potrzebuję dalszego leczenia stomatologicznego?",
        "ro": "Trebuie să plătesc din nou dacă am nevoie de tratament stomatologic suplimentar după câteva săptămâni?",
        "ar": "هل يجب أن أدفع مرة أخرى إذا احتجت إلى علاج أسنان إضافي بعد بضعة أسابيع؟",
        "ur": "اگر مجھے چند ہفتوں بعد دانتوں کا مزید علاج چاہیے تو کیا مجھے دوبارہ ادائیگی کرنی ہوگی؟"}),
    "M20": ("health", DENT, "your false teeth (dentures) need to be repaired",
            "No, repairing dentures is free. If they cannot be repaired, you pay for new ones.", {
        "en": "Do I have to pay to get my dentures repaired on the NHS?",
        "pl": "Czy muszę płacić za naprawę protezy zębowej w NHS?",
        "ro": "Trebuie să plătesc pentru repararea protezei dentare la NHS?",
        "ar": "هل يجب أن أدفع لإصلاح طقم أسناني لدى NHS؟",
        "ur": "کیا NHS میں ڈینچر (مصنوعی دانت) کی مرمت کے لیے مجھے ادائیگی کرنی ہوگی؟"}),
    "M21": ("health", PHARM, "take it to your pharmacy to be disposed of safely",
            "Do not put it in the bin or flush it away. Take it to a pharmacy to be disposed of safely.", {
        "en": "What should I do with old medicines I no longer need?",
        "pl": "Co zrobić ze starymi lekami, których już nie potrzebuję?",
        "ro": "Ce ar trebui să fac cu medicamentele vechi de care nu mai am nevoie?",
        "ar": "ماذا أفعل بالأدوية القديمة التي لم أعد بحاجة إليها؟",
        "ur": "مجھے پرانی دواؤں کا کیا کرنا چاہیے جن کی اب ضرورت نہیں؟"}),
    "M22": ("health", PREG, "maternity exemption (matex) certificate",
            "Yes. Ask your midwife or GP about the maternity exemption (MATEX) certificate, which gives free NHS "
            "prescriptions and free dental care.", {
        "en": "Can I get free prescriptions while I am pregnant?",
        "pl": "Czy w ciąży mogę dostawać leki na receptę za darmo?",
        "ro": "Pot primi medicamente gratuite pe rețetă cât sunt însărcinată?",
        "ar": "هل يمكنني الحصول على الأدوية الموصوفة مجانًا أثناء الحمل؟",
        "ur": "کیا حمل کے دوران مجھے نسخے کی دوائیں مفت مل سکتی ہیں؟"}),
    "M23": ("health", LIS, "£16,000 for everyone else",
            "Your savings, investments or property (not your home) must not be over £16,000, or £23,250 if you "
            "live permanently in a care home.", {
        "en": "What is the savings limit for the NHS Low Income Scheme?",
        "pl": "Jaki jest limit oszczędności w programie NHS Low Income Scheme?",
        "ro": "Care este limita de economii pentru NHS Low Income Scheme?",
        "ar": "ما هو حد المدخرات في برنامج NHS Low Income Scheme؟",
        "ur": "NHS لو انکم اسکیم کے لیے بچت کی حد کیا ہے؟"}),
    "M24": ("health", LIS, "complete an hc1 form",
            "Apply online, or by post using an HC1 form sent to the address on the form.", {
        "en": "How do I apply for the NHS Low Income Scheme by post?",
        "pl": "Jak złożyć wniosek do NHS Low Income Scheme pocztą?",
        "ro": "Cum aplic prin poștă pentru NHS Low Income Scheme?",
        "ar": "كيف أتقدم بطلب للانضمام إلى NHS Low Income Scheme عبر البريد؟",
        "ur": "میں ڈاک کے ذریعے NHS لو انکم اسکیم کے لیے کیسے درخواست دوں؟"}),
    "M25": ("health", ABROAD, "are exempt from charges",
            "Yes. A&E services are free to everyone, but emergency treatment after being admitted to hospital "
            "is not included.", {
        "en": "I am visiting England. Is A&E free for me?",
        "pl": "Jestem w Anglii z wizytą. Czy SOR (A&E) jest dla mnie darmowy?",
        "ro": "Sunt în vizită în Anglia. Este gratuit serviciul de urgențe (A&E) pentru mine?",
        "ar": "أنا في زيارة إلى إنجلترا. هل خدمة الطوارئ (A&E) مجانية بالنسبة لي؟",
        "ur": "میں انگلینڈ کے دورے پر ہوں۔ کیا A&E میرے لیے مفت ہے؟"}),
    "M26": ("health", ABROAD, "charged at 150% of the national nhs rate",
            "If you are not ordinarily resident and must pay for NHS hospital treatment, you are charged 150% "
            "of the national NHS rate.", {
        "en": "How much do overseas visitors pay for NHS hospital treatment?",
        "pl": "Ile płacą osoby przyjezdne z zagranicy za leczenie szpitalne w NHS?",
        "ro": "Cât plătesc vizitatorii din străinătate pentru tratamentul în spitalele NHS?",
        "ar": "كم يدفع الزوار من الخارج مقابل العلاج في مستشفيات NHS؟",
        "ur": "بیرون ملک سے آنے والے افراد NHS اسپتال کے علاج کے لیے کتنی ادائیگی کرتے ہیں؟"}),
    "M27": ("health", VACC, "around 20 weeks pregnant",
            "The whooping cough (pertussis) vaccine is offered around 20 weeks of pregnancy.", {
        "en": "When should I have the whooping cough vaccine in pregnancy?",
        "pl": "Kiedy w ciąży należy przyjąć szczepionkę przeciw krztuścowi?",
        "ro": "Când ar trebui să fac vaccinul împotriva tusei convulsive în sarcină?",
        "ar": "متى يجب أن أتلقى لقاح السعال الديكي أثناء الحمل؟",
        "ur": "حمل کے دوران کالی کھانسی کی ویکسین کب لگوانی چاہیے؟"}),
}

UNANSWERABLE = {
    "U11": {"en": "How do I apply for a Blue Badge for disabled parking?",
            "pl": "Jak ubiegać się o Blue Badge, czyli kartę parkingową dla osób niepełnosprawnych?",
            "ro": "Cum aplic pentru un Blue Badge pentru parcarea persoanelor cu dizabilități?",
            "ar": "كيف أتقدم بطلب للحصول على Blue Badge لمواقف ذوي الإعاقة؟",
            "ur": "معذور افراد کی پارکنگ کے لیے بلیو بیج (Blue Badge) کی درخواست کیسے دوں؟"},
    "U12": {"en": "How do I get a National Insurance number?",
            "pl": "Jak uzyskać numer National Insurance?",
            "ro": "Cum obțin un număr National Insurance?",
            "ar": "كيف أحصل على رقم التأمين الوطني (National Insurance)؟",
            "ur": "میں نیشنل انشورنس نمبر کیسے حاصل کروں؟"},
    "U13": {"en": "How much is Child Benefit per week?",
            "pl": "Ile wynosi tygodniowo Child Benefit (zasiłek na dziecko)?",
            "ro": "Cât este Child Benefit (alocația pentru copii) pe săptămână?",
            "ar": "كم تبلغ إعانة الطفل (Child Benefit) أسبوعيًا؟",
            "ur": "چائلڈ بینیفٹ ہفتہ وار کتنا ہے؟"},
    "U14": {"en": "How do I apply for a student loan for university?",
            "pl": "Jak złożyć wniosek o kredyt studencki na studia?",
            "ro": "Cum aplic pentru un împrumut pentru studenți pentru facultate?",
            "ar": "كيف أتقدم بطلب للحصول على قرض طلابي للدراسة الجامعية؟",
            "ur": "یونیورسٹی کے لیے اسٹوڈنٹ لون کی درخواست کیسے دوں؟"},
    "U15": {"en": "How do I report a pothole in my road?",
            "pl": "Jak zgłosić dziurę w jezdni na mojej ulicy?",
            "ro": "Cum raportez o groapă în carosabil pe strada mea?",
            "ar": "كيف أبلغ عن حفرة في الطريق في شارعي؟",
            "ur": "میں اپنی سڑک میں گڑھے کی اطلاع کیسے دوں؟"},
    "U16": {"en": "My rubbish bin was not collected this week. What should I do?",
            "pl": "W tym tygodniu nie odebrano moich śmieci. Co powinienem zrobić?",
            "ro": "Gunoiul meu nu a fost ridicat săptămâna aceasta. Ce ar trebui să fac?",
            "ar": "لم تُجمع حاوية القمامة الخاصة بي هذا الأسبوع. ماذا أفعل؟",
            "ur": "اس ہفتے میرا کوڑا نہیں اٹھایا گیا۔ مجھے کیا کرنا چاہیے؟"},
    "U17": {"en": "How do I apply for Carer's Allowance?",
            "pl": "Jak złożyć wniosek o Carer's Allowance (zasiłek dla opiekuna)?",
            "ro": "Cum aplic pentru Carer's Allowance (indemnizația de îngrijitor)?",
            "ar": "كيف أتقدم بطلب للحصول على بدل مقدم الرعاية (Carer's Allowance)؟",
            "ur": "کیئررز الاؤنس کے لیے درخواست کیسے دوں؟"},
    "U18": {"en": "How do I change my name by deed poll?",
            "pl": "Jak zmienić nazwisko za pomocą deed poll?",
            "ro": "Cum îmi schimb numele prin deed poll?",
            "ar": "كيف أغير اسمي عن طريق وثيقة deed poll؟",
            "ur": "میں ڈیڈ پول کے ذریعے اپنا نام کیسے تبدیل کروں؟"},
    "U19": {"en": "Can I get a free hearing aid on the NHS?",
            "pl": "Czy mogę dostać darmowy aparat słuchowy w NHS?",
            "ro": "Pot primi gratuit un aparat auditiv prin NHS?",
            "ar": "هل يمكنني الحصول على سماعة أذن طبية مجانية من NHS؟",
            "ur": "کیا مجھے NHS سے مفت سننے کا آلہ مل سکتا ہے؟"},
    "U20": {"en": "How long is statutory maternity leave?",
            "pl": "Jak długo trwa ustawowy urlop macierzyński?",
            "ro": "Cât durează concediul de maternitate legal?",
            "ar": "ما هي مدة إجازة الأمومة القانونية؟",
            "ur": "قانونی زچگی کی چھٹی کتنی طویل ہوتی ہے؟"},
}


def main():
    pages = {}
    for line in open(ROOT / "data/processed/pages.jsonl"):
        r = json.loads(line)
        pages[r["url"]] = " ".join(s["text"] for s in r["sections"]).lower()
    bad = [(q, url, ev) for q, (_, url, ev, _, _) in HELDOUT.items() if url not in pages or ev.lower() not in pages[url]]
    if bad:
        sys.exit("evidence not found on gold page:\n" + "\n".join(map(str, bad)))
    langs = ["en", "pl", "ro", "ar", "ur"]
    assert all(list(qs) == langs for *_, qs in HELDOUT.values()) and all(list(q) == langs for q in UNANSWERABLE.values())

    with open(ROOT / "eval/heldout_queries.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["qid", "domain", "lang", "question", "gold_url", "evidence", "translation_status"])
        for qid, (domain, url, ev, _, qs) in HELDOUT.items():
            for lang, text in qs.items():
                w.writerow([qid, domain, lang, text, url, ev, "source" if lang == "en" else "unverified"])
    with open(ROOT / "eval/heldout_unanswerable.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["qid", "lang", "question", "translation_status"])
        for qid, qs in UNANSWERABLE.items():
            for lang, text in qs.items():
                w.writerow([qid, lang, text, "source" if lang == "en" else "unverified"])

    # One row per topic (English question), the reference used by the scoring rubric.
    cal = list(csv.DictReader(open(ROOT / "eval/queries.csv", encoding="utf-8")))
    cal_un = list(csv.DictReader(open(ROOT / "eval/unanswerable.csv", encoding="utf-8")))
    rows = []
    for r in cal:
        if r["lang"] == "en":
            rows.append(["calibration", r["qid"], r["domain"], True, r["question"], r["gold_url"], r["evidence"],
                         CALIBRATION_REFERENCE[r["qid"]], "answer"])
    for r in cal_un:
        if r["lang"] == "en":
            rows.append(["calibration", r["qid"], "", False, r["question"], "", "", "", "refuse"])
    for qid, (domain, url, ev, ref, qs) in HELDOUT.items():
        rows.append(["heldout", qid, domain, True, qs["en"], url, ev, ref, "answer"])
    for qid, qs in UNANSWERABLE.items():
        rows.append(["heldout", qid, "", False, qs["en"], "", "", "", "refuse"])
    with open(ROOT / "eval/gold_set.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "qid", "domain", "answerable", "question_en", "gold_url", "evidence",
                    "reference_answer", "expected"])
        w.writerows(rows)
    n_ans = sum(r[3] for r in rows)
    print(f"gold set: {len(rows)} topics ({n_ans} answerable, {len(rows) - n_ans} unanswerable); "
          f"held-out: {len(HELDOUT)} answerable + {len(UNANSWERABLE)} unanswerable x 5 languages; "
          "all evidence phrases found verbatim on their gold page")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval/evaluate_week7.py
"""Week 7 experiment: held-out check of the search stage and refusal rules, and the ungrounded baseline.

Questions: the held-out half of the gold set (eval/build_gold_set.py), never used for tuning:
  50 answerable topics x 5 languages (eval/heldout_queries.csv)    = 250
  10 unanswerable topics x 5 languages (eval/heldout_unanswerable.csv) = 50

Steps
  1. Search every question with the week 5 pipeline (cached to results/week7_search.json) and measure
     hit@3 (gold page in the top 3) and ans@3 (a top-3 passage contains the evidence phrase), as in week 5.
  2. Grounded answers: the week 6 assistant exactly as it is (direct mode, threshold 0.0052 fixed in
     week 6, yes/no check, NO_ANSWER). Nothing is re-tuned, so the refusal figures are a fair test.
  3. Ungrounded baseline (T30): the same model, same style rules, no retrieved passages.
  4. Automatic measures for both conditions: refused, cited / cites_answer (grounded only), answer in the
     question's language (lingua), length, words per sentence, and Flesch Reading Ease / Flesch-Kincaid grade
     for English answers (textstat). Accuracy and unsupported claims are scored by hand with
     docs/scoring-rubric.md, using the blinded sheet written by step 5.
  5. Scoring sheet: 15 held-out English topics (12 answerable, 3 unanswerable, fixed random seed) x 2
     conditions, shuffled, condition hidden -> eval/scoring_sheet.xlsx; key in eval/scoring_key.csv.

Answers are cached row by row in results/week7_answers.csv, so the run can be resumed.
Run: python eval/evaluate_week7.py            (about 3 hours on an Apple-silicon laptop)
     python eval/evaluate_week7.py --quick    (8 topics x 5 languages)
Output: results/week7_*.csv, eval/scoring_sheet.xlsx, eval/scoring_key.csv
"""
import argparse
import json
import pathlib
import random
import re
import sys
import time

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import torch  # noqa: F401  (import before faiss on macOS, see retrieval.py)
import textstat
from language import detect
from pipeline import SearchPipeline
from generate import citations, CITATION
from translate import split_sentences

SEARCH_CACHE = ROOT / "results/week7_search.json"
ANSWERS = ROOT / "results/week7_answers.csv"
QUICK = ["S11", "S15", "H12", "H23", "M11", "M25", "U11", "U17"]


def norm(s):
    return " ".join(str(s).replace("’", "'").lower().split())


def load_questions():
    a = pd.read_csv(ROOT / "eval/heldout_queries.csv")
    a["answerable"] = True
    u = pd.read_csv(ROOT / "eval/heldout_unanswerable.csv")
    u["answerable"], u["evidence"], u["gold_url"], u["domain"] = False, "", "", ""
    return pd.concat([a, u], ignore_index=True)


def run_search(qs, pipe):
    cache = json.loads(SEARCH_CACHE.read_text()) if SEARCH_CACHE.exists() else {}
    for r in qs.itertuples():
        key = f"{r.qid}-{r.lang}"
        if key not in cache:
            cache[key] = pipe.search(r.question, k=3, lang=r.lang)
    SEARCH_CACHE.write_text(json.dumps(cache, ensure_ascii=False, default=str))
    return cache


def retrieval_table(qs, search):
    rows = []
    for r in qs[qs.answerable].itertuples():
        res = search[f"{r.qid}-{r.lang}"]["results"]
        rows.append({"qid": r.qid, "lang": r.lang, "hit@3": any(p["url"] == r.gold_url for p in res),
                     "ans@3": any(norm(r.evidence) in norm(p["text"]) for p in res),
                     "top_score": res[0]["score"]})
    return pd.DataFrame(rows)


def answer_text(answer):
    """The answer without citation markers and without the appended health note."""
    return CITATION.sub("", answer.split("\n\n")[0]).replace(" .", ".").strip()


def text_measures(text, lang):
    sents = [s for s in split_sentences(text) if s.strip()]
    words = len(text.split())
    out = {"words": words, "words_per_sentence": round(words / max(len(sents), 1), 1),
           "flesch_reading_ease": None, "fk_grade": None}
    if lang == "en" and words:
        out["flesch_reading_ease"] = round(textstat.flesch_reading_ease(text), 1)
        out["fk_grade"] = round(textstat.flesch_kincaid_grade(text), 1)
    return out


def scoring_sheet(df):
    """Blinded sheet for hand scoring (pilot and second rater): condition hidden, order shuffled."""
    gold = pd.read_csv(ROOT / "eval/gold_set.csv")
    gold = gold[gold.split == "heldout"]
    rng = random.Random(7)
    topics = rng.sample(sorted(gold[gold.answerable].qid), 12) + rng.sample(sorted(gold[~gold.answerable].qid), 3)
    en = df[(df.lang == "en") & df.qid.isin(topics)]
    items = [r for r in en.itertuples()]
    rng.shuffle(items)
    sheet, key = [], []
    for n, r in enumerate(items, 1):
        g = gold[gold.qid == r.qid].iloc[0]
        sheet.append({"item": n, "question": r.question,
                      "reference_answer": g.reference_answer if g.answerable else "(not covered by the 38 pages: should be refused)",
                      "source_page": g.gold_url if g.answerable else "", "answer_to_score": r.answer,
                      "accuracy": "", "unsupported_claim": "", "citations": "", "refusal": "", "notes": ""})
        key.append({"item": n, "qid": r.qid, "condition": r.condition})
    with pd.ExcelWriter(ROOT / "eval/scoring_sheet.xlsx") as xw:
        for rater in ("rater 1", "rater 2"):
            pd.DataFrame(sheet).to_excel(xw, sheet_name=rater, index=False)
    pd.DataFrame(key).to_csv(ROOT / "eval/scoring_key.csv", index=False)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true", help="answers for an 8-topic subset only")
    quick = ap.parse_args().quick
    from assistant import Assistant
    from generate import Generator

    qs = load_questions()
    if quick:
        qs = qs[qs.qid.isin(QUICK)].reset_index(drop=True)
    pipe = SearchPipeline()
    search = run_search(qs, pipe)
    ret = retrieval_table(qs, search)
    ret.to_csv(ROOT / "results/week7_retrieval.csv", index=False)
    print("held-out retrieval (answerable):")
    print(ret.groupby("lang")[["hit@3", "ans@3"]].mean().round(3).T.assign(ALL=ret[["hit@3", "ans@3"]].mean().round(3)).to_string())

    done = pd.read_csv(ANSWERS) if ANSWERS.exists() else pd.DataFrame()
    have = set(zip(done.qid, done.lang, done.condition)) if len(done) else set()
    rows = done.to_dict("records") if len(done) else []
    bot = Assistant(pipeline=pipe, generator=Generator(), mode="direct")
    jobs = [(r, c) for r in qs.itertuples() for c in ("grounded", "ungrounded") if (r.qid, r.lang, c) not in have]
    print(len(jobs), "answers to generate", flush=True)
    for n, (r, cond) in enumerate(jobs, 1):
        t0 = time.time()
        s = search[f"{r.qid}-{r.lang}"]
        if cond == "grounded":
            a = bot.answer_from(s, r.question, "direct")
            refused, reason, answer = a["refused"], a["reason"], a["answer"]
        else:
            refused, reason, answer = False, None, bot.generator.answer_ungrounded(r.question, r.lang)
        cited = citations(answer) if cond == "grounded" and not refused else []
        valid = bool(cited) and all(1 <= c <= len(s["results"]) for c in cited)
        cites_answer = bool(r.answerable) and any(norm(r.evidence) in norm(s["results"][c - 1]["text"])
                                                  for c in cited if 1 <= c <= len(s["results"]))
        text = answer_text(answer)
        rows.append({"qid": r.qid, "lang": r.lang, "condition": cond, "answerable": r.answerable,
                     "question": r.question, "top_score": s["results"][0]["score"], "refused": refused,
                     "reason": reason, "cited": valid, "cites_answer": cites_answer,
                     "lang_ok": (not refused) and detect(text) == r.lang,
                     **(text_measures(text, r.lang) if not refused else {}),
                     "answer": answer, "seconds": round(time.time() - t0, 1)})
        if n % 10 == 0 or n == len(jobs):
            pd.DataFrame(rows).to_csv(ANSWERS, index=False)
            print(f"{n}/{len(jobs)}", flush=True)

    df = pd.DataFrame(rows)
    df = df[df.set_index(["qid", "lang"]).index.isin(qs.set_index(["qid", "lang"]).index)]
    summarise(df)
    if not quick:
        scoring_sheet(df)


def summarise(df):
    out = []
    for cond in ("grounded", "ungrounded"):
        for lang in ["ALL", "en", "pl", "ro", "ar", "ur"]:
            d = df[(df.condition == cond) & ((df.lang == lang) | (lang == "ALL"))]
            a, u = d[d.answerable], d[~d.answerable]
            given = a[~a.refused]
            out.append({"condition": cond, "lang": lang, "n_answerable": len(a),
                        "answerable_refused": round(a.refused.mean(), 3),
                        "cited": round(given.cited.mean(), 3) if cond == "grounded" else None,
                        "cites_answer": round(given.cites_answer.mean(), 3) if cond == "grounded" else None,
                        "lang_ok": round(given.lang_ok.mean(), 3),
                        "words": round(given.words.mean(), 1),
                        "words_per_sentence": round(given.words_per_sentence.mean(), 1),
                        "flesch_reading_ease": round(given.flesch_reading_ease.mean(), 1) if lang in ("en", "ALL") else None,
                        "fk_grade": round(given.fk_grade.mean(), 1) if lang in ("en", "ALL") else None,
                        "n_unanswerable": len(u), "unanswerable_refused": round(u.refused.mean(), 3),
                        "sec_per_answer": round(d[~d.refused].seconds.mean(), 1)})
    by = pd.DataFrame(out)
    by.to_csv(ROOT / "results/week7_by_condition.csv", index=False)
    ref = (df[df.condition == "grounded"].groupby(["answerable", "reason"], dropna=False).size()
           .rename("n").reset_index())
    ref.to_csv(ROOT / "results/week7_refusal.csv", index=False)
    print(by.to_string(index=False))
    print(ref.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
!{sys.executable} src/ingest.py

In [ ]:
!{sys.executable} src/chunk.py

In [ ]:
!{sys.executable} eval/build_queries.py

In [ ]:
!{sys.executable} eval/build_unanswerable.py

In [ ]:
!{sys.executable} eval/build_gold_set.py

In [ ]:
!{sys.executable} -u eval/evaluate_week7.py --quick 2>&1 | grep -v -i -E 'warn|unauthenticated|Loading weights|falling back'

In [ ]:
import json, pathlib
import pandas as pd
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import torch
pd.set_option('display.max_colwidth', 120)

## 1. The gold set (T31)

100 question topics, each in 5 languages:

* **calibration** (40): the week 4-6 questions. The week 6 refusal threshold was set on these.
* **held-out** (60): new this week, never used to tune anything - 50 answerable, 10 unanswerable.

Every answerable topic has a gold page, an evidence phrase copied verbatim from that page (`eval/build_gold_set.py` stops if one is missing) and a short reference answer for hand scoring.

In [ ]:
gold = pd.read_csv(ROOT / 'eval/gold_set.csv')
print(gold.groupby(['split', 'answerable']).size().to_string())
gold[gold.split == 'heldout'][['qid', 'question_en', 'evidence', 'reference_answer']].head(10)

In [ ]:
pd.read_csv(ROOT / 'eval/heldout_unanswerable.csv').query("lang == 'en'")[['qid', 'question']]

## 2. Does the search stage hold up on new questions?

Same measures as week 5 on the 250 held-out answerable questions: **hit@3** (gold page in the top 3) and **ans@3** (a top-3 passage contains the evidence phrase). Week 5 on the calibration questions: hit@3 1.000, ans@3 0.893.

In [ ]:
ret = pd.read_csv(ROOT / 'results/week7_retrieval.csv')
t = ret.groupby('lang')[['hit@3', 'ans@3']].mean().loc[['en', 'pl', 'ro', 'ar', 'ur']]
t.loc['ALL'] = ret[['hit@3', 'ans@3']].mean()
t.round(3).T

## 3. Refusal rules on held-out questions (T28 re-checked)

The week 6 assistant is used exactly as it was: threshold 0.0052 on the top reranker score, the yes/no check, and NO_ANSWER. Nothing was re-tuned.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from assistant import THRESHOLD
ans = pd.read_csv(ROOT / 'results/week7_answers.csv')
g = ans[ans.condition == 'grounded']
bins = np.logspace(-5, 0, 30)
plt.figure(figsize=(8, 3.2))
plt.hist(g[g.answerable].top_score, bins=bins, alpha=0.7, label='held-out answerable (250)')
plt.hist(g[~g.answerable].top_score, bins=bins, alpha=0.7, label='held-out unanswerable (50)')
plt.axvline(THRESHOLD, color='k', ls='--', label=f'week 6 threshold {THRESHOLD}')
plt.xscale('log'); plt.xlabel('top reranker score (log scale)'); plt.ylabel('questions'); plt.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
pd.read_csv(ROOT / 'results/week7_refusal.csv')

In [ ]:
print('answerable questions refused:')
g[g.answerable & g.refused][['qid', 'lang', 'question', 'top_score', 'reason']]

In [ ]:
print('unanswerable questions answered:')
g[~g.answerable & ~g.refused][['qid', 'lang', 'top_score', 'answer']]

## 4. The ungrounded baseline (T30)

Same model (Qwen3.5-4B), same audience and plain-language rules, same language instruction - but no retrieved passages and no citations. It answers from what the model learned in training.

In [ ]:
from generate import baseline_messages
for m in baseline_messages('How quickly must my landlord put my deposit in a protection scheme?', 'en'):
    print(f"--- {m['role']} ---\n{m['content']}\n")

In [ ]:
def compare(qid, lang='en'):
    for r in ans[(ans.qid == qid) & (ans.lang == lang)].itertuples():
        print(f'[{r.condition}] {r.question}\n{r.answer}\n')
    ref = gold[gold.qid == qid].reference_answer.iloc[0]
    print(f'REFERENCE: {ref}\n' + '=' * 80)

for q in ['H12', 'M25', 'H25', 'U11', 'U17']:
    compare(q)

## 5. Automatic measures, grounded vs ungrounded

These check form, not truth: accuracy and unsupported claims are scored by hand (section 6). Readability: Flesch Reading Ease (higher = easier) and Flesch-Kincaid grade for English answers only; words per sentence for every language.

In [ ]:
by = pd.read_csv(ROOT / 'results/week7_by_condition.csv')
by.set_index(['condition', 'lang'])

## 6. Hand scoring: rubric and blinded sheet (T32)

`docs/scoring-rubric.md` defines four scores: accuracy (correct / partly correct / incorrect / refused), unsupported claim (yes / no), citations (all correct / some wrong / none / n/a) and refusal (correct refusal / answered anyway / n/a). `eval/scoring_sheet.xlsx` has 30 items (15 held-out English topics x 2 conditions), shuffled with the condition hidden, with a tab for each of two raters. Agreement is reported as percentage agreement and Cohen's kappa. The key is in `eval/scoring_key.csv`.

## 7. Findings (full local run, 16 Nov 2026)

1. **The search stage holds up on new questions.** On the 250 held-out answerable questions, hit@3 was 0.980 and
   ans@3 0.956 (week 5, calibration questions: 1.000 and 0.893). The held-out questions were written by the author
   after reading the pages, so they may be easier than real users' questions; this is a limitation, not a gain.
2. **The week 6 refusal rules mostly carry over without re-tuning.**
   * 2 of 250 answerable questions were wrongly refused (H20 Polish, M27 Romanian). In both, the answer passage was
     retrieved, but the top score (0.0037 and 0.0046) fell just under the threshold of 0.0052. In week 6, none of 150
     were refused.
   * 45 of 50 unanswerable questions were refused (week 6: 43 of 50): 30 by the threshold, 15 by the yes/no check.
   * Of the 5 that were answered, 3 are prose refusals ("the extracts do not contain…"). The other 2, both Urdu, are
     real errors:
     * U12 turns a passage about the IHS reference number into an answer about getting a National Insurance number;
     * U19 applies a general "help with NHS costs" passage to hearing aids.
3. **The ungrounded baseline never says it does not know.** It answered all 300 questions, including all 50
   unanswerable ones, although its prompt allows "say so". Compared with the reference answers from the official pages:
   * **H25:** it invents a "Child Tax Free Amount" of £210.50 a month (the page says £303.94).
   * **S11:** it says children cannot start school before their fifth birthday (the page: reception starts in the
     September after the fourth birthday).
   * **M25:** it adds a "visitor surcharge" of about £200.
   These are examples only; rates come from the hand scoring.
4. **Grounded answers cite their sources.**
   * 96.0% have valid citations.
   * 92.3% cite the passage that holds the answer phrase (week 6: 83.3%).
   * Urdu is the weakest language (0.920 and 0.860).
   * All answers in both conditions are in the question's language.
5. **The baseline reads more easily but has nothing behind it.** In English, the grounded answers score Flesch Reading
   Ease 66.2 and FK grade 8.4 (17.8 words per sentence). The ungrounded answers score 71.9 and grade 6.4 (13.2 words
   per sentence). Grounded answers keep the official wording, which has longer sentences.
6. **Hand scoring is prepared, not done.** `eval/scoring_sheet.xlsx` holds 30 blinded items for two raters. Accuracy,
   hallucination and citation-correctness figures will come from those scores, not from this notebook.
